# acomplete

> A high level unified function make api calls

In [ ]:
#| default_exp acomplete

## Imports

In [ ]:
#| export
import nbdev, yaml, json
from dataclasses import dataclass, field, fields
from importlib.resources import files
from fastcore.utils import *
from fastcore.meta import *
from fastspec.spec import *
from fastspec.oapi import *
from fastspec.errors import APIError

from fastllm.types import *
from fastllm.normalize import *
from fastllm.streaming import *

In [ ]:
#| hide
import base64
from fastcore.test import *
from lisette.core import lite_mk_func
from cachy.core import enable_cachy, disable_cachy, doms

In [ ]:
enable_cachy(doms=(*doms,'api.moonshot.ai'), hdrs=('content-type',))

In [ ]:
#| export
specs_path = files('fastllm') / 'specs'
ant_spec  = SpecParser.from_openapi(dict2obj(yaml.safe_load((specs_path/'anthropic.yml').read_text())))
oai_spec  = SpecParser.from_openapi(dict2obj(yaml.safe_load((specs_path/'openai.with-code-samples.yml').read_text())))
gem_spec  = SpecParser.from_discovery(dict2obj(json.loads((specs_path/'gemini.json').read_text())))

In [ ]:
specs_path.ls()

[Path('/Users/keremturgutlu/aai-ws/fastllm/fastllm/specs/anthropic.yml'), Path('/Users/keremturgutlu/aai-ws/fastllm/fastllm/specs/gemini.json'), Path('/Users/keremturgutlu/aai-ws/fastllm/fastllm/specs/openai.with-code-samples.yml'), Path('/Users/keremturgutlu/aai-ws/fastllm/fastllm/specs/spec_manifest.json'), Path('/Users/keremturgutlu/aai-ws/fastllm/fastllm/specs/openapi_docs.ipynb')]

In [ ]:
ant_cli = OpenAPIClient(ant_spec, headers={"x-api-key": os.environ["ANTHROPIC_API_KEY"], "anthropic-version": "2023-06-01"})
oai_cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"})
gem_cli = OpenAPIClient(gem_spec, headers={"x-goog-api-key": os.environ["GEMINI_API_KEY"]})
kimi_cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ['MOONSHOT_API_KEY']}"})
for op in kimi_cli.ops: op.base_url = 'https://api.moonshot.ai/v1'

In [ ]:
def simple_add(a:int,b:int):
    'add numbers'
    return a + b

In [ ]:
sch = lite_mk_func(simple_add);sch

{'type': 'function',
 'function': {'name': 'simple_add',
  'description': 'add numbers',
  'parameters': {'type': 'object',
   'properties': {'a': {'description': '', 'type': 'integer'},
    'b': {'description': '', 'type': 'integer'}},
   'required': ['a', 'b']}}}

## OpenAI Responses Denorm Msgs

Helpers to translate back to provider specific inputs from our internal `Msg` representation

In [ ]:
mn,inp = 'gpt-4o-mini','What is 3 + 5 and 10 + 20. Use the simple_add tool in parallel'
tools=[{"type": "function", **sch['function']}]
resp = await oai_cli.responses.create_response(model=mn, input=inp, tools=tools, stream=True)
async for comp in acollect_stream(resp, model=mn, api_name=ApiName.openai): pass

In [ ]:
def mk_user_msg(txt): return Msg(role='user', content=[Part(type=PartType.text, text=txt)])

In [ ]:
comp.tool_calls

[ToolCall(id='fc_02274252e499512b0069ea4c89c17c81a3b8c4dfb2377306fb', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'type': 'function_call', 'status': 'completed', 'call_id': 'call_cREBUUmJnPsQli12NcIkClUK'}),
 ToolCall(id='fc_02274252e499512b0069ea4c89c18c81a3817e953d471f6025', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'type': 'function_call', 'status': 'completed', 'call_id': 'call_2pObz739bx4aqOx4z5dcifc0'})]

In [ ]:
#| export
def mk_tool_res_msg(tool_calls:list[ToolCall], results:list[str|list]):
    'A util to prepare parallel tool call with str or media list results'
    parts = []
    for tc,res in zip(tool_calls, results):
        data = dict(id=tc.id, name=tc.name, arguments=tc.arguments, server=tc.server)
        parts.append(Part(type=PartType.tool_result, text=res, data=data))
    return Msg(role="tool", content=parts)

In [ ]:
trmsg = mk_tool_res_msg(comp.tool_calls, ('8', '30')); trmsg

Msg(role='tool', content=[Part(type=<PartType.tool_result: 'tool_result'>, text='8', data={'id': 'fc_02274252e499512b0069ea4c89c17c81a3b8c4dfb2377306fb', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False}), Part(type=<PartType.tool_result: 'tool_result'>, text='30', data={'id': 'fc_02274252e499512b0069ea4c89c18c81a3817e953d471f6025', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'server': False})], data=None)

In [ ]:
#| export
def denorm_openai_responses_tool_use(p:Part):
    "Convert canonical tool_use Part back to OpenAI Responses function_call item."
    return dict(type='function_call', call_id=p.data.get('id'), name=p.data.get('name'), arguments=json.dumps(p.data.get('arguments', '{}')))

In [ ]:
L(comp.message.content).map(denorm_openai_responses_tool_use)

[{'type': 'function_call', 'call_id': 'fc_02274252e499512b0069ea4c89c17c81a3b8c4dfb2377306fb', 'name': 'simple_add', 'arguments': '{"a": 3, "b": 5}'}, {'type': 'function_call', 'call_id': 'fc_02274252e499512b0069ea4c89c18c81a3817e953d471f6025', 'name': 'simple_add', 'arguments': '{"a": 10, "b": 20}'}]

This is not exported -- media tool results are added later

In [ ]:
def denorm_openai_responses_tool_result(p:Part):
    "Convert canonical tool result back to OpenAI Responses function_call_output item."
    return dict(type='function_call_output', call_id=p.data.get('id'), output=str(p.text))

In [ ]:
#| export
def denorm_openai_responses_tool(m:Msg):
    items = []
    for part in m.content:
        if part.type == PartType.tool_result: items.append(denorm_openai_responses_tool_result(part))
    return items

In [ ]:
denorm_openai_responses_tool(trmsg)

[{'type': 'function_call_output',
  'call_id': 'fc_02274252e499512b0069ea4c89c17c81a3b8c4dfb2377306fb',
  'output': '8'},
 {'type': 'function_call_output',
  'call_id': 'fc_02274252e499512b0069ea4c89c18c81a3817e953d471f6025',
  'output': '30'}]

This is not exported -- media inputs are added later

In [ ]:
def denorm_openai_responses_user(m:Msg):
    "Convert canonical user Msg to OpenAI Responses input message."
    parts = [{"type": "input_text", "text": p.text or ""} for p in m.content if p.type == PartType.text]
    return dict(type="message", role="user", content=parts)

In [ ]:
umsg = mk_user_msg(inp)
denorm_openai_responses_user(umsg)

{'type': 'message',
 'role': 'user',
 'content': [{'type': 'input_text',
   'text': 'What is 3 + 5 and 10 + 20. Use the simple_add tool in parallel'}]}

In [ ]:
#| export
def denorm_openai_responses_assistant(m:Msg):
    items, srv_call_id = [], None
    for p in m.content:
        if p.type == PartType.tool_use:
            items.append(denorm_openai_responses_tool_use(p))
            srv_call_id = p.data.get('id') if p.data.get('server') else None
        elif p.type in (PartType.text, PartType.thinking) and srv_call_id:
            items.append(dict(type='function_call_output', call_id=srv_call_id, output=p.text or ''))
            srv_call_id = None
        elif p.type in (PartType.text, PartType.thinking):
            items.append(dict(type="message", role="assistant", content=[dict(type="output_text", text=p.text)]))
    return items

In [ ]:
denorm_openai_responses_assistant(comp.message)

[{'type': 'function_call',
  'call_id': 'fc_02274252e499512b0069ea4c89c17c81a3b8c4dfb2377306fb',
  'name': 'simple_add',
  'arguments': '{"a": 3, "b": 5}'},
 {'type': 'function_call',
  'call_id': 'fc_02274252e499512b0069ea4c89c18c81a3817e953d471f6025',
  'name': 'simple_add',
  'arguments': '{"a": 10, "b": 20}'}]

In [ ]:
#| export
def denorm_openai_responses_msgs(msgs:list[Msg]):
    res = []
    for m in msgs:
        if m.role == 'user':        res.append(denorm_openai_responses_user(m))
        elif m.role == 'assistant': res.extend(denorm_openai_responses_assistant(m))
        elif m.role == 'tool':      res.extend(denorm_openai_responses_tool(m))
    return res

In [ ]:
inputs = denorm_openai_responses_msgs([umsg, comp.message, trmsg]); inputs

[{'type': 'message',
  'role': 'user',
  'content': [{'type': 'input_text',
    'text': 'What is 3 + 5 and 10 + 20. Use the simple_add tool in parallel'}]},
 {'type': 'function_call',
  'call_id': 'fc_02274252e499512b0069ea4c89c17c81a3b8c4dfb2377306fb',
  'name': 'simple_add',
  'arguments': '{"a": 3, "b": 5}'},
 {'type': 'function_call',
  'call_id': 'fc_02274252e499512b0069ea4c89c18c81a3817e953d471f6025',
  'name': 'simple_add',
  'arguments': '{"a": 10, "b": 20}'},
 {'type': 'function_call_output',
  'call_id': 'fc_02274252e499512b0069ea4c89c17c81a3b8c4dfb2377306fb',
  'output': '8'},
 {'type': 'function_call_output',
  'call_id': 'fc_02274252e499512b0069ea4c89c18c81a3817e953d471f6025',
  'output': '30'}]

In [ ]:
resp = await oai_cli.responses.create_response(model=mn, input=inputs, tools=tools, stream=True)
async for comp in acollect_stream(resp, model=mn, api_name=ApiName.openai): pass
comp.message.content[0].text

'The results are as follows:\n\n- \\(3 + 5 = 8\\)\n- \\(10 + 20 = 30\\)'

#### Text

In [ ]:
msg1 = mk_user_msg('Hi!')
resp = await oai_cli.responses.create_response(model=mn, input=denorm_openai_responses_msgs([msg1]), stream=True)
async for comp in acollect_stream(resp, model=mn, api_name=ApiName.openai): pass
print(comp.message.content[0].text)

Hello! How can I assist you today?


In [ ]:
denorm_openai_responses_msgs([msg1])

[{'type': 'message',
  'role': 'user',
  'content': [{'type': 'input_text', 'text': 'Hi!'}]}]

In [ ]:
msg2,msg3 = comp.message, mk_user_msg('How are you?')
resp = await oai_cli.responses.create_response(model=mn, input=denorm_openai_responses_msgs([msg1,msg2,msg3]), stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai): pass
print(resp_comp.message.content[0].text)

I'm doing well, thanks! How about you?


In [ ]:
denorm_openai_responses_msgs([msg1,msg2,msg3])

[{'type': 'message',
  'role': 'user',
  'content': [{'type': 'input_text', 'text': 'Hi!'}]},
 {'type': 'message',
  'role': 'assistant',
  'content': [{'type': 'output_text',
    'text': 'Hello! How can I assist you today?'}]},
 {'type': 'message',
  'role': 'user',
  'content': [{'type': 'input_text', 'text': 'How are you?'}]}]

#### Thinking

In [ ]:
mn = 'o4-mini'
msg1 = mk_user_msg('What is 31231231 * 12312?')
resp = await oai_cli.responses.create_response(model=mn, input=denorm_openai_responses_msgs([msg1]), reasoning={"effort": "low", "summary": "auto"}, stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai): pass
L(resp_comp.message.content).attrgot('type')

[<PartType.thinking: 'thinking'>, <PartType.thinking: 'thinking'>, <PartType.text: 'text'>]

In [ ]:
denorm_openai_responses_msgs([msg1])

[{'type': 'message',
  'role': 'user',
  'content': [{'type': 'input_text', 'text': 'What is 31231231 * 12312?'}]}]

In [ ]:
msg2,msg3 = resp_comp.message, mk_user_msg('Great!')
resp = await oai_cli.responses.create_response(model=mn, input=denorm_openai_responses_msgs([msg1,msg2,msg3]), reasoning={"effort": "low"}, stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai): pass
print(resp_comp.message.content[0].text)

Glad I could help! If you have any more questions or need further calculations, just let me know.


In [ ]:
denorm_openai_responses_msgs([msg1,msg2,msg3])

[{'type': 'message',
  'role': 'user',
  'content': [{'type': 'input_text', 'text': 'What is 31231231 * 12312?'}]},
 {'type': 'message',
  'role': 'assistant',
  'content': [{'type': 'output_text',
    'text': '**Calculating big multiplication**\n\nThe user wants me to multiply two large numbers: 31,231,231 and 12,312. First, I need to confirm the numbers I have. I decide to use long multiplication, breaking it down into manageable parts. \n\nI calculate A as 31,231,231 times 12,312 in two steps and find that A equals 374,774,772,000 and B equals 9,744,144,072. Adding these together, the total is 384,518,916,072.'}]},
 {'type': 'message',
  'role': 'assistant',
  'content': [{'type': 'output_text',
    'text': "**Verifying the calculation**\n\nI'm double-checking my previous multiplication, summing 374,774,772,000 and 9,744,144,072 to get 384,518,916,072, which looks correct. For additional verification, I check an approximate calculation: 31.23 million times 12.312 thousand gives abou

#### Tool Call

In [ ]:
mn = 'gpt-4o-mini'
tools=[{"type": "function", **sch['function']}]
msg1 = mk_user_msg('What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.')
resp = await oai_cli.responses.create_response(model=mn, input=denorm_openai_responses_msgs([msg1]), tools=tools, stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai): pass
L(resp_comp.message.content).attrgot('type')

[<PartType.tool_use: 'tool_use'>, <PartType.tool_use: 'tool_use'>]

In [ ]:
denorm_openai_responses_msgs([msg1])

[{'type': 'message',
  'role': 'user',
  'content': [{'type': 'input_text',
    'text': 'What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'}]}]

In [ ]:
trmsg = mk_tool_res_msg(resp_comp.tool_calls, ('8', '30')); trmsg

Msg(role='tool', content=[Part(type=<PartType.tool_result: 'tool_result'>, text='8', data={'id': 'fc_0e8ff00cea98b3be0069ea4c9e9b18819d8513fb3a1c562950', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False}), Part(type=<PartType.tool_result: 'tool_result'>, text='30', data={'id': 'fc_0e8ff00cea98b3be0069ea4c9e9b28819da97ad1d8ce89cc97', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'server': False})], data=None)

In [ ]:
msg2 = resp_comp.message
resp = await oai_cli.responses.create_response(model=mn, input=denorm_openai_responses_msgs([msg1,msg2,trmsg]), stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai): pass
print(resp_comp.message.content[0].text)

The results are:

- \(3 + 5 = 8\)
- \(10 + 20 = 30\)


In [ ]:
denorm_openai_responses_msgs([msg1,msg2,trmsg])

[{'type': 'message',
  'role': 'user',
  'content': [{'type': 'input_text',
    'text': 'What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'}]},
 {'type': 'function_call',
  'call_id': 'fc_0e8ff00cea98b3be0069ea4c9e9b18819d8513fb3a1c562950',
  'name': 'simple_add',
  'arguments': '{"a": 3, "b": 5}'},
 {'type': 'function_call',
  'call_id': 'fc_0e8ff00cea98b3be0069ea4c9e9b28819da97ad1d8ce89cc97',
  'name': 'simple_add',
  'arguments': '{"a": 10, "b": 20}'},
 {'type': 'function_call_output',
  'call_id': 'fc_0e8ff00cea98b3be0069ea4c9e9b18819d8513fb3a1c562950',
  'output': '8'},
 {'type': 'function_call_output',
  'call_id': 'fc_0e8ff00cea98b3be0069ea4c9e9b28819da97ad1d8ce89cc97',
  'output': '30'}]

#### Web Search

In [ ]:
msg1 = mk_user_msg('What is the weather in Istanbul today?')
tools=[{"type": "web_search_preview"}]
resp = await oai_cli.responses.create_response(model=mn, input=denorm_openai_responses_msgs([msg1]), tools=tools, stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai): pass
L(resp_comp.message.content).attrgot('type')

[<PartType.tool_use: 'tool_use'>, <PartType.text: 'text'>]

In [ ]:
resp_comp.message.content[1].text[:100]

'As of 7:45\u202fPM local time in Istanbul on April 23, 2026, the current weather is partly sunny with a t'

In [ ]:
denorm_openai_responses_msgs([msg1])

[{'type': 'message',
  'role': 'user',
  'content': [{'type': 'input_text',
    'text': 'What is the weather in Istanbul today?'}]}]

In [ ]:
msg2,msg3 = resp_comp.message, mk_user_msg('Great!')
resp = await oai_cli.responses.create_response(model=mn, input=denorm_openai_responses_msgs([msg1,msg2,msg3]), stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai): pass
print(resp_comp.message.content[0].text)

I'm glad you found the information helpful! If you have any more questions or need assistance with something else, feel free to ask. Enjoy your day!


In [ ]:
denorm_openai_responses_msgs([msg1,msg2,msg3])

[{'type': 'message',
  'role': 'user',
  'content': [{'type': 'input_text',
    'text': 'What is the weather in Istanbul today?'}]},
 {'type': 'function_call',
  'call_id': 'ws_0bd828ba4417947f0069ea4ca15a308196a4ea799f007b81ad',
  'name': 'web_search',
  'arguments': '{"type": "search", "queries": ["Istanbul weather today"], "query": "Istanbul weather today"}'},
 {'type': 'function_call_output',
  'call_id': 'ws_0bd828ba4417947f0069ea4ca15a308196a4ea799f007b81ad',
  'output': "As of 7:45\u202fPM local time in Istanbul on April 23, 2026, the current weather is partly sunny with a temperature of 53°F (12°C).\n\n## Weather for Hastane, Hadımköy-İstanbul Caddesi 34, 34555 Arnavutköy/İstanbul, Türkiye:\nCurrent Conditions: Partly sunny, 53°F (12°C)\n\nHourly Forecast:\n* 8:00\u202fPM: 54°F (12°C), Clear\n* 9:00\u202fPM: 52°F (11°C), Clear\n* 10:00\u202fPM: 49°F (10°C), Clear\n* 11:00\u202fPM: 44°F (7°C), Clear\n* 12:00\u202fAM: 47°F (8°C), Clear\n* 1:00\u202fAM: 49°F (9°C), Clear\n* 2:00\u

## OpenAI Chat Denorm Msgs

This is not exported -- media tool results are added later

In [ ]:
def denorm_openai_chat_tool_result(p:Part):
    "Convert canonical tool_result Part to OpenAI Chat tool message."
    return dict(role='tool', tool_call_id=p.data.get('id'), content=str(p.text))

This is not exported -- media inputs are added later

In [ ]:
def denorm_openai_chat_user(m:Msg):
    "Convert canonical user Msg to OpenAI Chat user message."
    if len(m.content) == 1 and m.content[0].type == PartType.text: return dict(role='user', content=m.content[0].text or '')
    return dict(role='user', content=[dict(type='text', text=p.text or '') for p in m.content if p.type == PartType.text])

In [ ]:
#| export
def denorm_openai_chat_tool_use(p:Part):
    "Convert canonical tool_use Part to OpenAI Chat tool_call dict."
    return dict(id=p.data.get('id'), type='function', function=dict(name=p.data.get('name'), arguments=json.dumps(p.data.get('arguments', '{}'))))

def denorm_openai_chat_assistant(m:Msg):
    "Convert canonical assistant Msg to OpenAI Chat assistant message + synthetic tool responses for server tools."
    tcs = [denorm_openai_chat_tool_use(p) for p in m.content if p.type == PartType.tool_use]
    msg, srv_responses, non_srv_texts, srv_call_id = dict(role='assistant'), [], [], None
    for p in m.content:
        if p.type == PartType.tool_use:
            srv_call_id = p.data.get('id') if p.data.get('server') else None
        elif p.type in (PartType.text, PartType.thinking) and srv_call_id:
            srv_responses.append(dict(role='tool', tool_call_id=srv_call_id, content=p.text or ''))
            srv_call_id = None
        elif p.type == PartType.text: non_srv_texts.append(p)
    if non_srv_texts: msg['content'] = non_srv_texts[0].text if len(non_srv_texts)==1 else [dict(type='text', text=p.text or '') for p in non_srv_texts]
    if tcs: msg['tool_calls'] = tcs
    thinking = [p for p in m.content if p.type == PartType.thinking]
    if thinking: msg['reasoning_content'] = ''.join(p.text or '' for p in thinking)
    return [msg] + srv_responses

def denorm_openai_chat_tool(m:Msg):
    "Convert canonical tool Msg to list of OpenAI Chat tool messages."
    return [denorm_openai_chat_tool_result(p) for p in m.content if p.type == PartType.tool_result]

def denorm_openai_chat_msgs(msgs:list[Msg]):
    "Convert list of canonical Msgs to OpenAI Chat messages."
    res = []
    for m in msgs:
        if   m.role == 'user':      res.append(denorm_openai_chat_user(m))
        elif m.role == 'assistant': res.extend(denorm_openai_chat_assistant(m))
        elif m.role == 'tool':      res.extend(denorm_openai_chat_tool(m))
    return res

#### Text

In [ ]:
mn='gpt-4o-mini'
msg1 = mk_user_msg('Hi!')
resp = await oai_cli.chat.create_chat_completion(model=mn,messages=denorm_openai_chat_msgs([msg1]),stream=True,stream_options={"include_usage": True})
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai_chat): pass
print(resp_comp.message.content[0].text)

Hello! How can I assist you today?


In [ ]:
denorm_openai_chat_msgs([msg1])

[{'role': 'user', 'content': 'Hi!'}]

In [ ]:
mn='gpt-4o-mini'
msg2,msg3 = resp_comp.message, mk_user_msg('How are you?')
resp = await oai_cli.chat.create_chat_completion(model=mn, messages=denorm_openai_chat_msgs([msg1,msg2,msg3]),stream=True,stream_options={"include_usage": True})
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai_chat): pass
print(resp_comp.message.content[0].text)

I'm just a computer program, but thanks for asking! How about you? How can I help you today?


In [ ]:
denorm_openai_chat_msgs([msg1,msg2,msg3])

[{'role': 'user', 'content': 'Hi!'},
 {'role': 'assistant', 'content': 'Hello! How can I assist you today?'},
 {'role': 'user', 'content': 'How are you?'}]

#### Thinking

In [ ]:
mn='kimi-k2.5'
msg1 = mk_user_msg('What is 31231231 * 12312?')
resp = await kimi_cli.chat.create_chat_completion(model=mn,messages=denorm_openai_chat_msgs([msg1]),stream=True,stream_options={"include_usage": True})
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai_chat): pass
L(resp_comp.message.content).attrgot('type')

[<PartType.thinking: 'thinking'>, <PartType.text: 'text'>]

In [ ]:
denorm_openai_chat_msgs([msg1])

[{'role': 'user', 'content': 'What is 31231231 * 12312?'}]

In [ ]:
msg2,msg3 = resp_comp.message, mk_user_msg('Great!')
resp = await kimi_cli.chat.create_chat_completion(model=mn, messages=denorm_openai_chat_msgs([msg1,msg2,msg3]),stream=True,stream_options={"include_usage": True})
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai_chat): pass
L(resp_comp.message.content).attrgot('type')

[<PartType.thinking: 'thinking'>, <PartType.text: 'text'>]

In [ ]:
resp_comp.message.content[1].text

"You're welcome! Glad I could help with that calculation. If you have any other math questions (or anything else), feel free to ask!"

In [ ]:
denorm_openai_chat_msgs([msg1,msg2,msg3])

[{'role': 'user', 'content': 'What is 31231231 * 12312?'},
 {'role': 'assistant',
  'content': "31231231 × 12312 = **384,518,916,072**\n\nHere's the breakdown:\n- 31,231,231 × 10,000 = 312,312,310,000\n- 31,231,231 × 2,000 = 62,462,462,000\n- 31,231,231 × 300 = 9,369,369,300\n- 31,231,231 × 10 = 312,312,310\n- 31,231,231 × 2 = 62,462,462\n\n**Sum:** 384,518,916,072",
  'reasoning_content': "The user is asking for the product of 31231231 and 12312. This is a straightforward multiplication problem, but the numbers are large enough that manual calculation might be error-prone. Let me calculate it step by step.\n\nFirst, let me break it down:\n31231231 × 12312\n\nI can use the distributive property:\n12312 = 12000 + 300 + 12\n\nSo:\n31231231 × 12312 = 31231231 × (12000 + 300 + 12)\n= 31231231 × 12000 + 31231231 × 300 + 31231231 × 12\n\nLet me calculate each part:\n\n1. 31231231 × 12000 = 31231231 × 12 × 1000\n   31231231 × 12:\n   - 31231231 × 10 = 312312310\n   - 31231231 × 2 = 62462462\n

#### Tool Call

In [ ]:
mn='gpt-4o-mini'
msg1 = mk_user_msg('What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.')
resp = await oai_cli.chat.create_chat_completion(model=mn,messages=denorm_openai_chat_msgs([msg1]), tools=[sch],stream=True,stream_options={"include_usage": True})
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai_chat): pass
print(resp_comp.message.content)

[Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'call_VGNQoGfoMeHB7jEeKNhvRkIX', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False, 'index': 0, 'type': 'function'}), Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'call_pwlG3QyTKHCx160PTmem5FKU', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'server': False, 'index': 1, 'type': 'function'})]


In [ ]:
denorm_openai_chat_msgs([msg1])

[{'role': 'user',
  'content': 'What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'}]

In [ ]:
mn='gpt-4o-mini'
msg2,msg3 = resp_comp.message, mk_tool_res_msg(resp_comp.tool_calls, ('8', '30'))
resp = await oai_cli.chat.create_chat_completion(model=mn, messages=denorm_openai_chat_msgs([msg1,msg2,msg3]),stream=True,stream_options={"include_usage": True})
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai_chat): pass
print(resp_comp.message.content[0].text)

The result of \(3 + 5\) is \(8\), and the result of \(10 + 20\) is \(30\).


In [ ]:
denorm_openai_chat_msgs([msg1,msg2,msg3])

[{'role': 'user',
  'content': 'What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'},
 {'role': 'assistant',
  'tool_calls': [{'id': 'call_VGNQoGfoMeHB7jEeKNhvRkIX',
    'type': 'function',
    'function': {'name': 'simple_add', 'arguments': '{"a": 3, "b": 5}'}},
   {'id': 'call_pwlG3QyTKHCx160PTmem5FKU',
    'type': 'function',
    'function': {'name': 'simple_add', 'arguments': '{"a": 10, "b": 20}'}}]},
 {'role': 'tool',
  'tool_call_id': 'call_VGNQoGfoMeHB7jEeKNhvRkIX',
  'content': '8'},
 {'role': 'tool',
  'tool_call_id': 'call_pwlG3QyTKHCx160PTmem5FKU',
  'content': '30'}]

## Anthropic Messages Denorm Msgs

In [ ]:
#| export
def _ant_cc(block, p):
    "Add cache_control to block if present in Part.data."
    if (cc := (p.data or {}).get('cache_control')): block['cache_control'] = cc
    return block

This is not exported -- media tool results are added later

In [ ]:
def denorm_anthropic_tool_result(p:Part):
    "Convert canonical tool_result Part to Anthropic tool_result content block."
    d = p.data or {}
    return _ant_cc(dict(type='tool_result', tool_use_id=d.get('id'), content=str(p.text)), p)

This is not exported -- media inputs are added later

In [ ]:
def denorm_anthropic_user(m:Msg):
    "Convert canonical user Msg to Anthropic user message."
    parts = [_ant_cc(dict(type='text', text=p.text or ''), p) for p in m.content if p.type == PartType.text]
    return dict(role='user', content=parts)

In [ ]:
#| export
def denorm_anthropic_tool_use(p:Part):
    "Convert canonical tool_use Part to Anthropic tool_use content block."
    d = p.data or {}
    block = dict(type='tool_use', id=d.get('id',''), name=d.get('name',''), input=d.get('arguments') or {})
    if 'caller' in d: block['caller'] = d['caller']
    return _ant_cc(block, p)

def denorm_anthropic_assistant(m:Msg):
    "Convert canonical assistant Msg to Anthropic assistant message + synthetic tool results for non-Anthropic server tools."
    blocks, srv_results, srv_call_id = [], [], None
    for p in m.content:
        if p.type == PartType.thinking:
            if srv_call_id:
                srv_results.append(dict(type='tool_result', tool_use_id=srv_call_id, content=p.text or ''))
                srv_call_id = None
            elif sig:=(p.data or {}).get('signature',''): blocks.append(dict(type='thinking', thinking=p.text or '', signature=sig))
            else:                               blocks.append(_ant_cc(dict(type='text', text=p.text or ''), p))
        elif p.type == PartType.text:
            if srv_call_id:
                srv_results.append(dict(type='tool_result', tool_use_id=srv_call_id, content=p.text or ''))
                srv_call_id = None
            else: blocks.append(_ant_cc(dict(type='text', text=p.text or ''), p))
        elif p.type == PartType.tool_use:
            if p.data.get('server') and (p.data.get('id','') or '').startswith('srvtoolu_'):
                blocks.append(dict(type='server_tool_use', id=p.data['id'], name=p.data.get('name',''), input=p.data.get('arguments') or {}))
            elif p.data.get('server'):
                blocks.append(denorm_anthropic_tool_use(p))
                srv_call_id = p.data.get('id','')
            else: blocks.append(denorm_anthropic_tool_use(p))
        elif p.type == PartType.server_tool_result: blocks.append(p.data if p.data else dict(type='server_tool_result'))
    res = [dict(role='assistant', content=blocks)]
    if srv_results: res.append(dict(role='user', content=srv_results))
    return res

def denorm_anthropic_tool(m:Msg):
    "Convert canonical tool Msg to Anthropic user message with tool_result blocks."
    blocks = [denorm_anthropic_tool_result(p) for p in m.content if p.type == PartType.tool_result]
    return [dict(role='user', content=blocks)]

def denorm_anthropic_msgs(msgs:list[Msg]):
    "Convert list of canonical Msgs to Anthropic messages."
    res = []
    for m in msgs:
        if   m.role == 'user':      res.append(denorm_anthropic_user(m))
        elif m.role == 'assistant': res.extend(denorm_anthropic_assistant(m))
        elif m.role == 'tool':      res.extend(denorm_anthropic_tool(m))
    return res

#### Text

In [ ]:
mn,msg1 = 'claude-sonnet-4-20250514', mk_user_msg('Say hi')
think,hdrs = {"type": "enabled", "budget_tokens": 5000}, {"anthropic-version": "2023-06-01"}
resp = await ant_cli.messages.messages_post(model=mn,messages=denorm_anthropic_msgs([msg1]),max_tokens=8000,thinking=think, headers=hdrs,stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.anthropic): pass
print(resp_comp.message.content)

[Part(type=<PartType.thinking: 'thinking'>, text='The user is asking me to say hi, which is a simple greeting. I should respond in a friendly and welcoming way.', data={'citations': []}), Part(type=<PartType.text: 'text'>, text='Hi there! How are you doing today?', data={'citations': []})]


In [ ]:
denorm_anthropic_msgs([msg1])

[{'role': 'user', 'content': [{'type': 'text', 'text': 'Say hi'}]}]

In [ ]:
msg2,msg3 = resp_comp.message, mk_user_msg('Great!')
resp = await ant_cli.messages.messages_post(model=mn,messages=denorm_anthropic_msgs([msg1,msg2,msg3]),max_tokens=8000,thinking=think, headers=hdrs,stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.anthropic): pass
print(resp_comp.message.content)

[Part(type=<PartType.thinking: 'thinking'>, text='The user responded positively to my greeting, saying "Great!" This is a friendly, upbeat response. I should match their positive energy and maybe ask a follow-up question or offer to help with something.', data={'citations': []}), Part(type=<PartType.text: 'text'>, text="That's wonderful to hear! Is there anything I can help you with today?", data={'citations': []})]


In [ ]:
denorm_anthropic_msgs([msg1,msg2,msg3])

[{'role': 'user', 'content': [{'type': 'text', 'text': 'Say hi'}]},
 {'role': 'assistant',
  'content': [{'type': 'text',
    'text': 'The user is asking me to say hi, which is a simple greeting. I should respond in a friendly and welcoming way.'},
   {'type': 'text', 'text': 'Hi there! How are you doing today?'}]},
 {'role': 'user', 'content': [{'type': 'text', 'text': 'Great!'}]}]

#### Tool Call

In [ ]:
mn,msg1 = 'claude-sonnet-4-20250514', mk_user_msg('What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.')
think,hdrs = {"type": "enabled", "budget_tokens": 5000}, {"anthropic-version": "2023-06-01"}
tools = [{"name": "simple_add", "description": "add numbers", "input_schema": sch['function']['parameters']}]
resp = await ant_cli.messages.messages_post(model=mn, messages=denorm_anthropic_msgs([msg1]), max_tokens=8000, tools=tools, headers=hdrs, stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.anthropic): pass
print(resp_comp.message.content)

[Part(type=<PartType.text: 'text'>, text="I'll calculate both additions for you using the simple_add tool in parallel.", data={'citations': []}), Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'toolu_01EikbsrTRTmPF3cP4e5TRoC', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False, 'caller': {'type': 'direct'}}), Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'toolu_01MxkXTJbaczc7Wi5UimQ7xA', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'server': False, 'caller': {'type': 'direct'}})]


In [ ]:
denorm_anthropic_msgs([msg1])

[{'role': 'user',
  'content': [{'type': 'text',
    'text': 'What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'}]}]

In [ ]:
msg2,msg3 = resp_comp.message, mk_tool_res_msg(resp_comp.tool_calls, ('8', '30'))
resp = await ant_cli.messages.messages_post(model=mn,messages=denorm_anthropic_msgs([msg1,msg2,msg3]),max_tokens=8000,thinking=think, headers=hdrs,stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.anthropic): pass
print(resp_comp.message.content[0].text)

The results are:
- 3 + 5 = 8
- 10 + 20 = 30


In [ ]:
denorm_anthropic_msgs([msg1,msg2,msg3])

[{'role': 'user',
  'content': [{'type': 'text',
    'text': 'What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'}]},
 {'role': 'assistant',
  'content': [{'type': 'text',
    'text': "I'll calculate both additions for you using the simple_add tool in parallel."},
   {'type': 'tool_use',
    'id': 'toolu_01EikbsrTRTmPF3cP4e5TRoC',
    'name': 'simple_add',
    'input': {'a': 3, 'b': 5},
    'caller': {'type': 'direct'}},
   {'type': 'tool_use',
    'id': 'toolu_01MxkXTJbaczc7Wi5UimQ7xA',
    'name': 'simple_add',
    'input': {'a': 10, 'b': 20},
    'caller': {'type': 'direct'}}]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01EikbsrTRTmPF3cP4e5TRoC',
    'content': '8'},
   {'type': 'tool_result',
    'tool_use_id': 'toolu_01MxkXTJbaczc7Wi5UimQ7xA',
    'content': '30'}]}]

#### Web Search

In [ ]:
mn,msg1 = 'claude-sonnet-4-20250514', mk_user_msg('Use web search to get the weather in Istanbul?')
think,hdrs = {"type": "enabled", "budget_tokens": 5000}, {"anthropic-version": "2023-06-01"}
tools=[{"type": "web_search_20250305", "name": "web_search"}]
resp = await ant_cli.messages.messages_post(model=mn, messages=denorm_anthropic_msgs([msg1]), tools=tools, max_tokens=8000, thinking=think, headers=hdrs, stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.anthropic): pass
L(resp_comp.message.content).attrgot('type')

[<PartType.thinking: 'thinking'>, <PartType.tool_use: 'tool_use'>, <PartType.server_tool_result: 'server_tool_result'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>]

In [ ]:
denorm_anthropic_msgs([msg1])

[{'role': 'user',
  'content': [{'type': 'text',
    'text': 'Use web search to get the weather in Istanbul?'}]}]

In [ ]:
msg2,msg3 = resp_comp.message, mk_user_msg('Great!')
resp = await ant_cli.messages.messages_post(model=mn,messages=denorm_anthropic_msgs([msg1,msg2,msg3]),max_tokens=8000,thinking=think, headers=hdrs,stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.anthropic): pass
print(resp_comp.message.content)

[Part(type=<PartType.thinking: 'thinking'>, text='The user has responded positively to my weather report for Istanbul. They seem satisfied with the information I provided. This is a simple acknowledgment, so I should respond appropriately - acknowledging their satisfaction and perhaps offering to help with anything else they might need.', data={'citations': []}), Part(type=<PartType.text: 'text'>, text="You're welcome! I'm glad I could help you get the current weather information for Istanbul. If you need any other weather updates, forecasts for different cities, or have any other questions, feel free to ask!", data={'citations': []})]


## Gemini Generate Denorm Msgs

This is not exported -- media tool results are added later

In [ ]:
def denorm_gemini_tool_result(p:Part):
    "Convert canonical tool_result Part to Gemini functionResponse part."
    d = p.data or {}
    fr = dict(name=d.get('name',''), response={"content": str(p.text)})
    if d.get('id'): fr['id'] = d['id']
    return dict(functionResponse=fr)

This is not exported -- media inputs are added later

In [ ]:
def denorm_gemini_user(m:Msg):
    "Convert canonical user Msg to Gemini user content."
    parts = [dict(text=p.text or '') for p in m.content if p.type == PartType.text]
    return dict(role='user', parts=parts)

In [ ]:
#| export
def denorm_gemini_tool_use(p:Part):
    "Convert canonical tool_use Part to Gemini functionCall part."
    d = p.data or {}
    fc = dict(name=d.get('name',''), args=d.get('arguments') or {})
    if d.get('id'): fc['id'] = d['id']
    part = dict(functionCall=fc)
    part['thoughtSignature'] = d.get('thoughtSignature', 'skip_thought_signature_validator')
    return part

def denorm_gemini_assistant(m:Msg):
    "Convert canonical assistant Msg to Gemini model content."
    parts = []
    for p in m.content:
        if   p.type == PartType.thinking: parts.append(dict(text=p.text or '', thought=True))
        elif p.type == PartType.text:     parts.append(dict(text=p.text or ''))
        elif p.type == PartType.tool_use: parts.append(denorm_gemini_tool_use(p))
    return dict(role='model', parts=parts)

def denorm_gemini_tool(m:Msg):
    "Convert canonical tool Msg to Gemini user content with functionResponse parts."
    parts = [denorm_gemini_tool_result(p) for p in m.content if p.type == PartType.tool_result]
    return dict(role='user', parts=parts)

def denorm_gemini_msgs(msgs:list[Msg]):
    "Convert list of canonical Msgs to Gemini contents."
    res = []
    for m in msgs:
        if   m.role == 'user':      res.append(denorm_gemini_user(m))
        elif m.role == 'assistant': res.append(denorm_gemini_assistant(m))
        elif m.role == 'tool':      res.append(denorm_gemini_tool(m))
    return res

#### Text

In [ ]:
mn,msg1 = 'models/gemini-3-flash-preview',mk_user_msg('Hi how are you?')
resp = await gem_cli.models.stream_generate_content(model=mn, contents=denorm_gemini_msgs([msg1]), stream=True, _query={"alt": "sse"})
async for comp in acollect_stream(resp, model=mn, api_name=ApiName.gemini): pass
print(comp.message.content[0].text)

I'm doing great, thank you for asking! How are you doing today? Is there anything I can help you with?


In [ ]:
denorm_gemini_msgs([msg1])

[{'role': 'user', 'parts': [{'text': 'Hi how are you?'}]}]

In [ ]:
msg2,msg3 = comp.message, mk_user_msg('Great!')
resp = await gem_cli.models.stream_generate_content(model=mn, contents=denorm_gemini_msgs([msg1,msg2,msg3]), stream=True, _query={"alt": "sse"})
async for comp in acollect_stream(resp, model=mn, api_name=ApiName.gemini): pass
print(comp.message.content[0].text)

That's wonderful to hear! I'm glad you're having a good day. 

What's on your mind? Is there anything specific you'd like to chat about or anything I can help you with today?


In [ ]:
denorm_gemini_msgs([msg1,msg2,msg3])

[{'role': 'user', 'parts': [{'text': 'Hi how are you?'}]},
 {'role': 'model',
  'parts': [{'text': "I'm doing great, thank you for asking! How are you doing today? Is there anything I can help you with?"}]},
 {'role': 'user', 'parts': [{'text': 'Great!'}]}]

#### Thinking

In [ ]:
mn,msg1 = 'models/gemini-3-flash-preview',mk_user_msg('What is 31231231 * 12312?')
resp = await gem_cli.models.stream_generate_content(model=mn, contents=denorm_gemini_msgs([msg1]), generation_config={"thinking_config": {"include_thoughts": True}}, stream=True, _query={"alt": "sse"})
async for comp in acollect_stream(resp, model=mn, api_name=ApiName.gemini): pass
comp.message.content

[Part(type=<PartType.thinking: 'thinking'>, text="**Calculating the Product**\n\nI'm currently breaking down the multiplication of 31,231,231 and 12,312. My approach involves simplifying the calculation. I've started by considering multiplying 31,231,231 by 12,000, as an initial step. This feels more manageable, and I'll adjust the subsequent calculations from there.\n\n\n**Verifying the Result**\n\nI've completed the step-by-step calculation and have double-checked it using standard long multiplication. The final answer, 384,518,916,072, aligns perfectly. I'm satisfied that my method of breaking down the multiplication into manageable steps, specifically by grouping and calculating in steps, has been successful.\n\n\n", data={'citations': []}),
 Part(type=<PartType.text: 'text'>, text='31,231,231 × 12,312 = **384,518,916,072**', data={'citations': []})]

In [ ]:
denorm_gemini_msgs([msg1])

[{'role': 'user', 'parts': [{'text': 'What is 31231231 * 12312?'}]}]

In [ ]:
msg2,msg3 = comp.message, mk_user_msg('Great!')
resp = await gem_cli.models.stream_generate_content(model=mn, contents=denorm_gemini_msgs([msg1,msg2,msg3]), generation_config={"thinking_config": {"include_thoughts": True}}, stream=True, _query={"alt": "sse"})
async for comp in acollect_stream(resp, model=mn, api_name=ApiName.gemini): pass
comp.message.content

[Part(type=<PartType.text: 'text'>, text="You're welcome! Do you have any other calculations or questions I can help you with?", data={'citations': []})]

In [ ]:
denorm_gemini_msgs([msg1,msg2,msg3])

[{'role': 'user', 'parts': [{'text': 'What is 31231231 * 12312?'}]},
 {'role': 'model',
  'parts': [{'text': "**Calculating the Product**\n\nI'm currently breaking down the multiplication of 31,231,231 and 12,312. My approach involves simplifying the calculation. I've started by considering multiplying 31,231,231 by 12,000, as an initial step. This feels more manageable, and I'll adjust the subsequent calculations from there.\n\n\n**Verifying the Result**\n\nI've completed the step-by-step calculation and have double-checked it using standard long multiplication. The final answer, 384,518,916,072, aligns perfectly. I'm satisfied that my method of breaking down the multiplication into manageable steps, specifically by grouping and calculating in steps, has been successful.\n\n\n",
    'thought': True},
   {'text': '31,231,231 × 12,312 = **384,518,916,072**'}]},
 {'role': 'user', 'parts': [{'text': 'Great!'}]}]

#### Tool Call

In [ ]:
msg1 = mk_user_msg('What is 3 + 5 and 10 + 20. Use the simple_add tool in parallel')
resp = await gem_cli.models.stream_generate_content(model=mn, contents=denorm_gemini_msgs([msg1]), tools=[{"functionDeclarations": [sch['function']]}], stream=True, _query={"alt": "sse"})
async for comp in acollect_stream(resp, model=mn, api_name=ApiName.gemini): pass
comp.message.content

[Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'vqsad6br', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False, 'thoughtSignature': 'EpoDCpcDAQw51sd9OVxWmdRgFwx/M8qEDamTogFLgS86j5+508TfVWhjgPshKVtmyOVeqz9TRk8HrV+zMtTyJJZXLWNnf3njWhGFva2N0fe0xejcTFxWBpRiyLjZonfitjj7baYIuJf50Umpb4tJah8G1GGrOj2dFM6XDSTL/YyffQ63f6aKUSmmCY4UgcYJ2wkoDNQcfL4wK8/6NZQ2hmZG+MhZTaBrYKiD6GN1y1Djnfl8anoJMqWwBn6za3JBmS2QUzDztBfJRXRmYJUMEfgDXOzAepHzIkhQ6UZcMZZSXyCKZXoH2HO22+bUmFsAlgEQtE4Bg5qdJusxxv71GSMV9yi+38RRd6FB+kX2XkK2xQI3dgZZpISnU79oI9bM5G0c5rnax+z74CPztoAekI/pU1pRZKUEJBeNA0THBUFdPBHBAcPkj5hHSay62MLdzxUPHCfZx95g76AWkMQdkKRhzxJgjjMUYrVu8CwhPRodJzD8FF0BWPv2fE+63nRX7PThn0YdjVTJFXlNaBm5/gR/YQzwrbwkuPlv1+E='}),
 Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'swkgb5j4', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'server': False})]

In [ ]:
denorm_openai_responses_msgs([msg1])

[{'type': 'message',
  'role': 'user',
  'content': [{'type': 'input_text',
    'text': 'What is 3 + 5 and 10 + 20. Use the simple_add tool in parallel'}]}]

In [ ]:
msg2, trmsg = comp.message, mk_tool_res_msg(comp.tool_calls, ('8','30'))
resp = await gem_cli.models.stream_generate_content(model=mn, contents=denorm_gemini_msgs([msg1,msg2,trmsg]), tools=[{"functionDeclarations": [sch['function']]}], stream=True, _query={"alt": "sse"})
async for comp in acollect_stream(resp, model=mn, api_name=ApiName.gemini): pass
comp.message.content

[Part(type=<PartType.text: 'text'>, text='3 + 5 is 8, and 10 + 20 is 30.', data={'citations': []})]

We use a valid `skip_thought_signature_validator` to skip thought signatures:

In [ ]:
denorm_gemini_msgs([msg1,msg2,trmsg])

[{'role': 'user',
  'parts': [{'text': 'What is 3 + 5 and 10 + 20. Use the simple_add tool in parallel'}]},
 {'role': 'model',
  'parts': [{'functionCall': {'name': 'simple_add',
     'args': {'a': 3, 'b': 5},
     'id': 'vqsad6br'},
    'thoughtSignature': 'EpoDCpcDAQw51sd9OVxWmdRgFwx/M8qEDamTogFLgS86j5+508TfVWhjgPshKVtmyOVeqz9TRk8HrV+zMtTyJJZXLWNnf3njWhGFva2N0fe0xejcTFxWBpRiyLjZonfitjj7baYIuJf50Umpb4tJah8G1GGrOj2dFM6XDSTL/YyffQ63f6aKUSmmCY4UgcYJ2wkoDNQcfL4wK8/6NZQ2hmZG+MhZTaBrYKiD6GN1y1Djnfl8anoJMqWwBn6za3JBmS2QUzDztBfJRXRmYJUMEfgDXOzAepHzIkhQ6UZcMZZSXyCKZXoH2HO22+bUmFsAlgEQtE4Bg5qdJusxxv71GSMV9yi+38RRd6FB+kX2XkK2xQI3dgZZpISnU79oI9bM5G0c5rnax+z74CPztoAekI/pU1pRZKUEJBeNA0THBUFdPBHBAcPkj5hHSay62MLdzxUPHCfZx95g76AWkMQdkKRhzxJgjjMUYrVu8CwhPRodJzD8FF0BWPv2fE+63nRX7PThn0YdjVTJFXlNaBm5/gR/YQzwrbwkuPlv1+E='},
   {'functionCall': {'name': 'simple_add',
     'args': {'a': 10, 'b': 20},
     'id': 'swkgb5j4'},
    'thoughtSignature': 'skip_thought_signature_validator'}]},
 {'role': 'user',
  'p

#### Web Search

In [ ]:
msg1 = mk_user_msg('What is the weather in Istanbul today?')
resp = await gem_cli.models.stream_generate_content(model=mn, contents=denorm_gemini_msgs([msg1]), tools=[{"googleSearch": {}}], stream=True, _query={"alt": "sse"})
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.gemini): pass
print(resp_comp.message.content)

[Part(type=<PartType.text: 'text'>, text='Today in Istanbul, it is a clear evening with a temperature of **53°F (12°C)**.\n\nEarlier in the day, the weather was mostly sunny with a high of approximately **57°F (14°C)**. The forecast for the remainder of tonight remains clear, with temperatures expected to drop to a low of around **47°F (8°C)**.\n\n**Weather Details for Today (April 23, 2026):**\n*   **Conditions:** Sunny day / Clear night\n*   **High Temperature:** 57°F (14°C)\n*   **Low Temperature:** 47°F (8°C)\n*   **Humidity:** 61%\n*   **Chance of Rain:** Minimal (0-20%)\n\nIf you are planning for tomorrow, Friday, April 24, expect slightly warmer weather with mostly sunny skies and a high of **61°F (16°C)**.', data={'citations': [{'searchEntryPoint': {'renderedContent': '<style>\n.container {\n  align-items: center;\n  border-radius: 8px;\n  display: flex;\n  font-family: Google Sans, Roboto, sans-serif;\n  font-size: 14px;\n  line-height: 20px;\n  padding: 8px 12px;\n}\n.chip {\

In [ ]:
denorm_openai_responses_msgs([msg1])

[{'type': 'message',
  'role': 'user',
  'content': [{'type': 'input_text',
    'text': 'What is the weather in Istanbul today?'}]}]

In [ ]:
msg2,msg3 = resp_comp.message, mk_user_msg('Great!')
resp = await gem_cli.models.stream_generate_content(model=mn, contents=denorm_gemini_msgs([msg1,msg2,msg3]), tools=[{"googleSearch": {}}], stream=True, _query={"alt": "sse"})
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.gemini): pass
print(resp_comp.message.content)

[Part(type=<PartType.text: 'text'>, text='Glad to hear it! If you need any more information—whether it’s more weather updates, travel tips for Istanbul, or anything else—just let me know. Have a wonderful day!', data={'citations': []})]


In [ ]:
denorm_openai_responses_msgs([msg1,msg2,msg3])

## acomplete

For our test I think we first need to provide a canonicalized way (e.g. openai chat format) and have their provider specific denorm functions for `tools`, `tool_choice`, `max_tokens`, `temperature`, `reasoning_effort` wdyt?

Sure

Good call

### Tool Schemas

In [ ]:
#| export
def _fn_schema(t):
    "Extract (name, description, parameters) from any tool format."
    if not isinstance(t, dict): return None
    fn = t.get('function')
    if isinstance(fn, dict): return fn.get('name',''), fn.get('description',''), fn.get('parameters',{})
    if 'name' in t and ('parameters' in t or 'input_schema' in t):
        return t.get('name',''), t.get('description',''), t.get('parameters', t.get('input_schema',{}))
    return None

def denorm_openai_responses_tools(tools):
    "Convert canonical tools to OpenAI Responses format."
    out = []
    for t in tools:
        fn = _fn_schema(t)
        if fn is None: out.append(t); continue
        name, desc, params = fn
        out.append(dict(type='function', name=name, description=desc, parameters=params))
    return out

def denorm_openai_chat_tools(tools):
    "Passthrough — canonical format is already OpenAI Chat."
    return tools

def denorm_anthropic_tools(tools):
    "Convert canonical tools to Anthropic format."
    out = []
    for t in tools:
        fn = _fn_schema(t)
        if fn is None: out.append(t); continue
        name, desc, params = fn
        out.append(dict(name=name, description=desc, input_schema=params))
    return out

def denorm_gemini_tools(tools):
    "Convert canonical tools to Gemini format."
    fn_decls, other = [], []
    for t in tools:
        fn = _fn_schema(t)
        if fn is None: other.append(t); continue
        name, desc, params = fn
        fn_decls.append(dict(name=name, description=desc, parameters=params))
    out = other[:]
    if fn_decls: out.insert(0, dict(functionDeclarations=fn_decls))
    return out

Add tests for each provider with a user defined function schema, and non-function tools

In [ ]:
sch

In [ ]:
# Canonical tool (OpenAI Chat format) + non-function tools per provider
fn_tool = sch  # {"type": "function", "function": {"name": "simple_add", ...}}

# OpenAI Responses
test_eq(denorm_openai_responses_tools([fn_tool]), [{'type': 'function', 'name': 'simple_add', 'description': 'add numbers', 'parameters': sch['function']['parameters']}])
test_eq(denorm_openai_responses_tools([{"type": "web_search_preview"}]), [{"type": "web_search_preview"}])

# OpenAI Chat (passthrough)
test_eq(denorm_openai_chat_tools([fn_tool]), [fn_tool])

# Anthropic
test_eq(denorm_anthropic_tools([fn_tool]), [{'name': 'simple_add', 'description': 'add numbers', 'input_schema': sch['function']['parameters']}])
test_eq(denorm_anthropic_tools([{"type": "web_search_20250305", "name": "web_search"}]), [{"type": "web_search_20250305", "name": "web_search"}])

# Gemini
test_eq(denorm_gemini_tools([fn_tool]), [{'functionDeclarations': [{'name': 'simple_add', 'description': 'add numbers', 'parameters': sch['function']['parameters']}]}])
test_eq(denorm_gemini_tools([{"googleSearch": {}}]), [{"googleSearch": {}}])
test_eq(denorm_gemini_tools([fn_tool, {"googleSearch": {}}]), [{'functionDeclarations': [{'name': 'simple_add', 'description': 'add numbers', 'parameters': sch['function']['parameters']}]}, {"googleSearch": {}}])

### Tool Choice

.

Yes, write helpers here and add tests

what about specifying a tool name in tool_choice?

Yes please, before that see if there are other things you are missing that we can support in canonical args?

> Dict passthrough: provider-native dicts pass through as-is

explain this please

Let's keep it simple. We will anyway think about how to expose the following next:

```
native: dict = None
extra_body: dict = None
extra_headers: dict = None
extra_query: dict = None
```

In [ ]:
#| export
_tc_modes = {'auto', 'required', 'any', 'force', 'none', 'off', 'disabled'}

def denorm_openai_responses_tool_choice(v):
    "Map canonical tool_choice to OpenAI Responses format."
    if v is None: return None
    s = str(v).strip().lower()
    if s in _tc_modes: return s if s in ('auto','none','required') else {'auto':'auto','any':'required','force':'required','off':'none','disabled':'none'}[s]
    return {'type': 'function', 'name': v}

def denorm_openai_chat_tool_choice(v):
    "Map canonical tool_choice to OpenAI Chat format."
    if v is None: return None
    s = str(v).strip().lower()
    if s in _tc_modes: return s if s in ('auto','none','required') else {'any':'required','force':'required','off':'none','disabled':'none'}[s]
    return {'type': 'function', 'function': {'name': v}}

def denorm_anthropic_tool_choice(v):
    "Map canonical tool_choice to Anthropic format."
    if v is None: return None
    s = str(v).strip().lower()
    if s in ('auto',):                    return {'type': 'auto'}
    if s in ('required', 'any', 'force'): return {'type': 'any'}
    if s in ('none', 'off', 'disabled'):  return None
    return {'type': 'tool', 'name': v}

def denorm_gemini_tool_choice(v):
    "Map canonical tool_choice to Gemini toolConfig."
    if v is None: return None
    s = str(v).strip().lower()
    if s in ('auto',):                    return {'functionCallingConfig': {'mode': 'AUTO'}}
    if s in ('required', 'any', 'force'): return {'functionCallingConfig': {'mode': 'ANY'}}
    if s in ('none', 'off', 'disabled'):  return {'functionCallingConfig': {'mode': 'NONE'}}
    return {'functionCallingConfig': {'mode': 'ANY', 'allowedFunctionNames': [v]}}

In [ ]:
# Modes
test_eq(denorm_openai_responses_tool_choice('auto'), 'auto')
test_eq(denorm_openai_responses_tool_choice('required'), 'required')
test_eq(denorm_openai_chat_tool_choice('required'), 'required')
test_eq(denorm_anthropic_tool_choice('auto'), {'type': 'auto'})
test_eq(denorm_anthropic_tool_choice('required'), {'type': 'any'})
test_eq(denorm_anthropic_tool_choice('none'), None)
test_eq(denorm_gemini_tool_choice('auto'), {'functionCallingConfig': {'mode': 'AUTO'}})
test_eq(denorm_gemini_tool_choice('none'), {'functionCallingConfig': {'mode': 'NONE'}})

# Tool name
test_eq(denorm_openai_responses_tool_choice('simple_add'), {'type': 'function', 'name': 'simple_add'})
test_eq(denorm_openai_chat_tool_choice('simple_add'), {'type': 'function', 'function': {'name': 'simple_add'}})
test_eq(denorm_anthropic_tool_choice('simple_add'), {'type': 'tool', 'name': 'simple_add'})
test_eq(denorm_gemini_tool_choice('simple_add'), {'functionCallingConfig': {'mode': 'ANY', 'allowedFunctionNames': ['simple_add']}})

# None
test_eq(denorm_openai_responses_tool_choice(None), None)
test_eq(denorm_anthropic_tool_choice(None), None)
test_eq(denorm_gemini_tool_choice(None), None)

### Reasoning Effort

No `disable` option maybe add later if needed. Anthropic uses the newer adaptive thinking which will error with legacy models < 4.6

Ok let's do other args now

In [ ]:
#| export
def denorm_openai_responses_reasoning(v):
    "Map canonical reasoning_effort to OpenAI Responses reasoning param."
    if v is None: return None
    return {'effort': v} if isinstance(v, str) else v

def denorm_openai_chat_reasoning(v): return v  # passthrough as reasoning_effort param

_ant_think_aliases = dict(minimal='low', low='low', medium='medium', high='high', xhigh='xhigh', max='max')

def denorm_anthropic_reasoning(v):
    "Map canonical reasoning_effort to Anthropic adaptive thinking + output_config."
    if v is None: return None
    e = _ant_think_aliases.get(str(v).lower(), 'high')
    return {"thinking": {"type": "adaptive"}, "output_config": {"effort": e}}

_gem_think_budgets = dict(minimal=128, low=1024, medium=2048, high=4096)
_gem_think_levels  = dict(minimal='low', low='low', medium='medium', high='high')

def denorm_gemini_reasoning(v, model=''):
    "Map canonical reasoning_effort to Gemini thinkingConfig (uses thinkingLevel for Gemini 3+)."
    if v is None: return None
    if isinstance(v, dict): return v
    k = str(v).lower()
    # defaults to includeThoughts same as litellm
    if 'gemini-3' in model: return {'thinkingLevel': _gem_think_levels.get(k, 'medium'), 'includeThoughts': True}
    return {'thinkingBudget': _gem_think_budgets.get(str(v).lower(), 1024), 'includeThoughts': True}

In [ ]:
# reasoning_effort
test_eq(denorm_openai_responses_reasoning('low'), {'effort': 'low'})
test_eq(denorm_openai_responses_reasoning(None), None)
test_eq(denorm_openai_chat_reasoning('low'), 'low')
test_eq(denorm_anthropic_reasoning('low'), {"thinking": {"type": "adaptive"}, "output_config": {"effort": "low"}})
test_eq(denorm_anthropic_reasoning('high'), {"thinking": {"type": "adaptive"}, "output_config": {"effort": "high"}})
test_eq(denorm_anthropic_reasoning(None), None)
test_eq(denorm_gemini_reasoning('medium', 'models/gemini-2.5-flash'), {'thinkingBudget': 2048, 'includeThoughts': True})
test_eq(denorm_gemini_reasoning('low', 'models/gemini-3-flash-preview'), {'thinkingLevel': 'low', 'includeThoughts': True})
test_eq(denorm_gemini_reasoning('high', 'models/gemini-3-flash-preview'), {'thinkingLevel': 'high', 'includeThoughts': True})
test_eq(denorm_gemini_reasoning(None), None)

### Web Search Options

In [ ]:
#| export
_ant_search_max_uses = {"low": 1, "medium": 5, "high": 10}

def denorm_openai_chat_web_search(v): return v  # passthrough

def denorm_openai_responses_web_search(v):
    "Map canonical web_search_options to OpenAI Responses web_search_preview tool."
    t = {"type": "web_search_preview"}
    if (s := (v or {}).get("search_context_size")): t["search_context_size"] = s
    if (u := (v or {}).get("user_location")): t["user_location"] = u
    return t

def denorm_anthropic_web_search(v):
    "Map canonical web_search_options to Anthropic hosted web_search tool."
    t = {"type": "web_search_20260209", "name": "web_search"}
    if (s := (v or {}).get("search_context_size")):
        t["max_uses"] = _ant_search_max_uses.get(s, 5)
    if (u := (v or {}).get("user_location", {}).get("approximate")):
        t["user_location"] = {"type": "approximate", **u}
    return t

def denorm_gemini_web_search(v): return {"googleSearch": {}}

.

if stream it should yield all responses that we collect with `acollect`

It should use the non-stream path. You can see example non-stream calls in previous dialogs. As for cli detection let's build them on the fly for now and make provider ane explicit arg to the function. Later we can infer it 

Yes it should return directly for non streaming

How does litellm handle it in `acomplete`?

Ok so we just return `acollect_stream(resp, model=model, provider=p)`?

Sounds good

### acomplete

In [ ]:
def _sys_text(system):
    "Extract text from system (str or Part)."
    if system is None: return None
    return system if isinstance(system, str) else system.text

async def acomplete(msgs, model, api_name, stream=False, system=None, max_tokens=None, temperature=None, tools=None, tool_choice=None, reasoning_effort=None, **kwargs):
    "Unified completion across providers."
    if api_name == ApiName.openai:
        cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"})
        payload = dict(model=model, input=denorm_openai_responses_msgs(msgs), stream=stream)
        if system:           payload['instructions'] = _sys_text(system)
        if max_tokens:       payload['max_output_tokens'] = max_tokens
        if temperature is not None: payload['temperature'] = temperature
        if tools:            payload['tools'] = denorm_openai_responses_tools(tools)
        if tool_choice:      payload['tool_choice'] = denorm_openai_responses_tool_choice(tool_choice)
        if reasoning_effort: payload['reasoning'] = denorm_openai_responses_reasoning(reasoning_effort)
        resp = await cli.responses.create_response(**payload, **kwargs)
        if stream: return acollect_stream(resp, model=model, api_name=api_name)
        return normalize_openai_response(resp, model=model, api_name=api_name)

    elif api_name == ApiName.openai_chat:
        if 'kimi' in model.lower():
            cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ['MOONSHOT_API_KEY']}"})
            for op in cli.ops: op.base_url = 'https://api.moonshot.ai/v1'
        else:
            cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"})
        payload = dict(model=model, messages=denorm_openai_chat_msgs(msgs), stream=stream)
        if system:           payload['messages'].insert(0, {'role': 'system', 'content': _sys_text(system)})
        if stream: payload['stream_options'] = {"include_usage": True}
        if max_tokens:       payload['max_tokens'] = max_tokens
        if temperature is not None: payload['temperature'] = temperature
        if tools:            payload['tools'] = denorm_openai_chat_tools(tools)
        if tool_choice:      payload['tool_choice'] = denorm_openai_chat_tool_choice(tool_choice)
        if reasoning_effort: payload['reasoning_effort'] = denorm_openai_chat_reasoning(reasoning_effort)
        resp = await cli.chat.create_chat_completion(**payload, **kwargs)
        if stream: return acollect_stream(resp, model=model, api_name=api_name)
        return normalize_openai_chat_completion(resp, model=model, api_name=api_name)

    elif api_name == ApiName.anthropic:
        cli = OpenAPIClient(ant_spec, headers={"x-api-key": os.environ["ANTHROPIC_API_KEY"], "anthropic-version": "2023-06-01"})
        payload = dict(model=model, messages=denorm_anthropic_msgs(msgs), max_tokens=max_tokens or 1024, stream=stream)
        if system:
            if isinstance(system, Part):
                block = {'type': 'text', 'text': system.text or ''}
                if (cc := (system.data or {}).get('cache_control')): block['cache_control'] = cc
                payload['system'] = [block]
            else: payload['system'] = system
        if temperature is not None: payload['temperature'] = temperature
        if tools:            payload['tools'] = denorm_anthropic_tools(tools)
        tc = denorm_anthropic_tool_choice(tool_choice)
        if tc:               payload['tool_choice'] = tc
        think = denorm_anthropic_reasoning(reasoning_effort)
        if think: payload.update(think)
        resp = await cli.messages.messages_post(**payload, **kwargs)
        if stream: return acollect_stream(resp, model=model, api_name=api_name)
        return normalize_anthropic_message(resp, model=model, api_name=api_name)

    elif api_name == ApiName.gemini:
        cli = OpenAPIClient(gem_spec, headers={"x-goog-api-key": os.environ["GEMINI_API_KEY"]})
        gen_config = {}
        if max_tokens:       gen_config['maxOutputTokens'] = max_tokens
        if temperature is not None: gen_config['temperature'] = temperature
        think = denorm_gemini_reasoning(reasoning_effort,model)
        if think:            gen_config['thinkingConfig'] = think
        payload = dict(model=model, contents=denorm_gemini_msgs(msgs))
        if system:           payload['system_instruction'] = {'parts': [{'text': _sys_text(system)}]}
        if gen_config:       payload['generation_config'] = gen_config
        if tools:
            gem_tools = denorm_gemini_tools(tools)
            payload['tools'] = gem_tools
            has_fn = any('functionDeclarations' in t for t in gem_tools)
            has_srv = any(k in t for t in gem_tools for k in ('googleSearch','codeExecution','googleSearchRetrieval'))
            if has_fn and has_srv:
                payload.setdefault('tool_config', {})['includeServerSideToolInvocations'] = True
        tc = denorm_gemini_tool_choice(tool_choice)
        if tc:               payload.setdefault('tool_config', {}).update(tc)
        op = 'stream_generate_content' if stream else 'generate_content'
        if stream:           payload.update(stream=True, _query={"alt": "sse"})
        resp = await getattr(cli.models, op)(**payload, **kwargs)
        if stream: return acollect_stream(resp, model=model, api_name=api_name)
        return normalize_gemini_generate(resp, model=model, api_name=api_name)

For each text/thinking/tool call/web search add test that will make a call with 1 provider and follow with another, so a double for loop - 16 combinations in each case:

```py
for start_prov in [provider.openai, provider.chat,...]:
    for cont_prov in [provider.openai, provider.chat,...]
```

This will test whether model swapping logic works for all paths

> Note: web search tests use stream=True for the first call since acollect_stream handles server tool collation, t

what do you mean?

Exactly, but we would like to use only streaming in these tests, as it's a more popular choice

In [ ]:
provs = [ApiName.openai, ApiName.openai_chat, ApiName.anthropic, ApiName.gemini]
models = {ApiName.openai: 'gpt-4o-mini', ApiName.openai_chat: 'gpt-4o-mini', 
          ApiName.anthropic: 'claude-sonnet-4-6', ApiName.gemini: 'models/gemini-3-flash-preview'}
think_models = {ApiName.openai: 'o4-mini', ApiName.openai_chat: 'kimi-k2.5',
                ApiName.anthropic: 'claude-sonnet-4-6', ApiName.gemini: 'models/gemini-3-flash-preview'}

async def stream_complete(msgs, model, prov, **kw):
    async for comp in await acomplete(msgs, model=model, api_name=prov, stream=True, **kw): pass
    return comp

#### Text - 16 combos
print("=== TEXT ===")
msg1 = Msg(role='user', content=[Part(type=PartType.text, text='Say hi in one word')])
first = {sp: await stream_complete([msg1], model=models[sp], prov=sp) for sp in provs}
for sp in provs:
    for cp in provs:
        msg2, msg3 = first[sp].message, Msg(role='user', content=[Part(type=PartType.text, text='Now say bye in one word')])
        comp = await stream_complete([msg1, msg2, msg3], model=models[cp], prov=cp)
        print(f'  {sp.value:12} -> {cp.value:12}: {comp.message.content[0].text[:60]}')

#### Thinking - 16 combos
print("\n=== THINKING ===")
msg1 = Msg(role='user', content=[Part(type=PartType.text, text='What is 123 * 456?')])
first_t = {sp: await stream_complete([msg1], model=think_models[sp], prov=sp, reasoning_effort='low') for sp in provs}
for sp in provs:
    for cp in provs:
        msg2, msg3 = first_t[sp].message, Msg(role='user', content=[Part(type=PartType.text, text='Now what is 789 * 2?')])
        comp = await stream_complete([msg1, msg2, msg3], model=think_models[cp], prov=cp, reasoning_effort='low')
        types = L(comp.message.content).attrgot('type')
        print(f'  {sp.value:12} -> {cp.value:12}: parts={types} {comp.message.content[0].text[:60]}')

#### Tool Call - 16 combos
print("\n=== TOOL CALL ===")
fn_tool = sch
msg1 = Msg(role='user', content=[Part(type=PartType.text, text='What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.')])
first_tc = {sp: await stream_complete([msg1], model=models[sp], prov=sp, tools=[fn_tool]) for sp in provs}
for sp in provs:
    for cp in provs:
        msg2 = first_tc[sp].message
        tmsg = mk_tool_res_msg(first_tc[sp].tool_use_parts, ('8','30'))
        comp = await stream_complete([msg1, msg2, tmsg], model=models[cp], prov=cp)
        print(f'  {sp.value:12} -> {cp.value:12}: {comp.message.content[0].text[:60]}')

#### Web Search - 12 combos (3 start x 4 continue)
print("\n=== WEB SEARCH ===")
ws_tools = {ApiName.openai: [{"type": "web_search_preview"}],
            ApiName.anthropic: [{"type": "web_search_20250305", "name": "web_search"}],
            ApiName.gemini: [{"googleSearch": {}}]}
ws_provs = [ApiName.openai, ApiName.anthropic, ApiName.gemini]
msg1 = Msg(role='user', content=[Part(type=PartType.text, text='What is the weather in Istanbul?')])
first_ws = {sp: await stream_complete([msg1], model=models[sp], prov=sp, tools=ws_tools[sp]) for sp in ws_provs}
for sp in ws_provs:
    for cp in provs:
        msg2, msg3 = first_ws[sp].message, Msg(role='user', content=[Part(type=PartType.text, text='Thanks!')])
        comp = await stream_complete([msg1, msg2, msg3], model=models[cp], prov=cp)
        print(f'  {sp.value:12} -> {cp.value:12}: {comp.message.content[0].text[:60]}')

## Other Features

### System

In [ ]:
msg1 = Msg(role='user', content=[Part(type=PartType.text, text='What are you?')])
sys_prompt = 'You are a pirate. Always respond in pirate speak. Keep it to one sentence.'
for p in provs:
    comp = await stream_complete([msg1], model=models[p], prov=p, system=sys_prompt)
    print(f'  {p.value:12}: {comp.message.content[0].text[:80]}')

### Caching

- **No canonical `cache` option** — caching is handled per-provider
- **Anthropic**: users put `cache_control` in `Part.data` — e.g. `Part(type="text", text=..., data={"cache_control": {"type": "ephemeral"}})`
- **OpenAI**: automatic, or via `native={"store": True}` - native escape hatch not yet implemented
- **Gemini**: automatic, or via `native={"cachedContent": "..."}` - native escape hatch not yet implemented

System prompt caching

In [ ]:
cc = {"cache_control": {"type": "ephemeral"}}
sys_part = Part(type=PartType.text, text='You are a helpful pirate assistant. ' * 200, data=cc)
msg1 = Msg(role='user', content=[Part(type=PartType.text, text='Say hi')])
comp = await stream_complete([msg1], model='claude-sonnet-4-20250514', prov=ApiName.anthropic, system=sys_part)
print(f'  System: {comp.usage.raw}')

User message caching (large context in first message)

In [ ]:
big_text = 'The quick brown fox jumps over the lazy dog. ' * 200
msg1 = Msg(role='user', content=[Part(type=PartType.text, text=big_text, data=cc), Part(type=PartType.text, text='Summarize')])
comp1 = await stream_complete([msg1], model='claude-sonnet-4-20250514', prov=ApiName.anthropic)
comp1.usage.raw

In [ ]:
msg2, msg3 = comp1.message, Msg(role='user', content=[Part(type=PartType.text, text='Now in French')])
comp2 = await stream_complete([msg1, msg2, msg3], model='claude-sonnet-4-20250514', prov=ApiName.anthropic)
test_eq(comp2.usage.raw.get('cache_read_input_tokens', 0) > 0, True)
comp2.usage.raw

Tool result caching

In [ ]:
tool_res_part = Part(type=PartType.tool_result, text='The answer is 42. ' * 200, data={**cc, 'id': 'toolu_test', 'name': 'compute'})
tmsg = Msg(role='tool', content=[tool_res_part])
msg1 = Msg(role='user', content=[Part(type=PartType.text, text='Compute the answer')])
msg2 = Msg(role='assistant', content=[Part(type=PartType.tool_use, data={'id': 'toolu_test', 'name': 'compute', 'arguments': {}, 'server': False})])
msg3 = Msg(role='user', content=[Part(type=PartType.text, text='Explain')])
comp = await stream_complete([msg1, msg2, tmsg, msg3], model='claude-sonnet-4-20250514', prov=ApiName.anthropic)
comp.usage.raw

### Media Inputs

`fastllm` canonicalizes multimodal inputs into `Part(type, text, data)` where `text` carries the URL or data URL, and `data` is reserved for optional metadata. Each provider's denorm layer converts this canonical form into the provider's wire format.

**Canonical Part Types**

| `Part.type` | `Part.text` | Description |
|---|---|---|
| `input_image` | URL or `data:image/...;base64,...` | Image input — JPEG, PNG, GIF, WEBP |
| `input_audio` | `data:audio/...;base64,...` | Audio input — WAV, MP3 (base64 only for OpenAI) |
| `input_video` | URL | Video input — MP4, WebM (Gemini only) |
| `input_file` | URL or `data:application/...;base64,...` | Document input — PDF, DOCX, TXT, etc. |

**Provider Wire Formats**

**OpenAI Responses** ([InputImageContent](https://developers.openai.com/api/reference/resources/responses/methods/create), [InputFileContent](https://developers.openai.com/api/reference/resources/responses/methods/create) — `specs/openai.with-code-samples.yml:68361,68429`):
- Image: `{"type": "input_image", "image_url": url_or_data_url}`
- File: `{"type": "input_file", "file_url": url}` or `{"type": "input_file", "file_data": data_url, "filename": "..."}`
- Audio/Video: not supported

**OpenAI Chat** ([ContentPartImage](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create), [ContentPartAudio](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create), [ContentPartFile](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create) — `specs/openai.with-code-samples.yml:35185,35217,35253`):
- Image: `{"type": "image_url", "image_url": {"url": url_or_data_url}}`
- Audio: `{"type": "input_audio", "input_audio": {"data": b64, "format": "wav"|"mp3"}}` — base64 only, WAV/MP3 only
- File: `{"type": "file", "file": {"file_data": data_url, "filename": "..."}}` — base64/file_id only, no URL
- Video: not supported

**Anthropic** ([RequestImageBlock](https://docs.anthropic.com/en/api/messages), [RequestDocumentBlock](https://docs.anthropic.com/en/api/messages) — `specs/anthropic.yml:12159,6740`):
- Image: `{"type": "image", "source": {"type": "url", "url": ...}}` or `{"type": "image", "source": {"type": "base64", "media_type": mime, "data": b64}}`
- File: `{"type": "document", "source": ...}` — same source pattern, PDF only (`media_type: "application/pdf"`)
- Audio/Video: not supported
- Supports `cache_control` on all content blocks

**Gemini** ([Part/Blob/FileData](https://ai.google.dev/api/generate-content) — `specs/gemini.json:189–387`):
- All media: `{"inlineData": {"mimeType": mime, "data": b64}}` or `{"fileData": {"mimeType": mime, "fileUri": url}}`
- Broadest format support — images, audio, video (incl. YouTube URLs), documents
- Supports `videoMetadata` for video start/end offsets

**Provider Support Matrix**

| Canonical type | OpenAI Responses | OpenAI Chat | Anthropic | Gemini |
|---|---|---|---|---|
| `input_image` (URL) | ✅ | ✅ | ✅ | ✅ |
| `input_image` (base64) | ✅ | ✅ | ✅ | ✅ |
| `input_audio` (base64) | ❌ | ✅ (WAV/MP3) | ❌ | ✅ |
| `input_video` (URL) | ❌ | ❌ | ❌ | ✅ |
| `input_file` (URL) | ✅ | ❌ | ✅ | ✅ |
| `input_file` (base64) | ✅ | ✅ | ✅ | ✅ |


Let's ignore `Part.data` if it will be only needed by optional extras, now update the mime dict, and write the code again

In [ ]:
#| export
_ext_mime = {
    '.jpg':'image/jpeg', '.jpeg':'image/jpeg', '.png':'image/png', '.gif':'image/gif', '.webp':'image/webp',
    '.pdf':'application/pdf',
    '.mp3':'audio/mpeg', '.wav':'audio/wav', '.ogg':'audio/ogg', '.flac':'audio/flac', '.m4a':'audio/mp4',
    '.mp4':'video/mp4', '.mov':'video/quicktime', '.webm':'video/webm',
}

def _data_url(url):
    "Parse data:mime;base64,data URL into (mime, b64_data), or None."
    if not isinstance(url, str) or not url.startswith('data:') or ',' not in url: return None
    header, body = url.split(',', 1)
    if ';base64' not in header or not body: return None
    return header[5:].split(';',1)[0].strip() or 'application/octet-stream', body

def _url_mime(url, default='application/octet-stream'):
    "Guess mime from URL extension."
    ext = '.' + url.rsplit('.', 1)[-1].split('?')[0].lower() if '.' in url.split('?')[0].split('/')[-1] else ''
    return _ext_mime.get(ext, default)

In [ ]:
#| export
def denorm_openai_responses_user(m:Msg):
    "Convert canonical user Msg to OpenAI Responses input message."
    parts = []
    for p in m.content:
        if   p.type == PartType.text:        parts.append({"type": "input_text", "text": p.text or ""})
        elif p.type == PartType.input_image: parts.append(_denorm_oai_resp_image(p))
        elif p.type == PartType.input_audio: raise ValueError("OpenAI Responses API does not support audio input; Coming Soon.") 
        elif p.type == PartType.input_video: raise ValueError("OpenAI Responses API does not support video input")
        elif p.type == PartType.input_file:  parts.append(_denorm_oai_resp_file(p))
    return dict(type='message', role='user', content=parts)

def denorm_openai_chat_user(m:Msg):
    "Convert canonical user Msg to OpenAI Chat user message."
    parts = []
    for p in m.content:
        if   p.type == PartType.text:        parts.append({"type": "text", "text": p.text or ""})
        elif p.type == PartType.input_image: parts.append(_denorm_oai_chat_image(p))
        elif p.type == PartType.input_audio: parts.append(_denorm_oai_chat_audio(p))
        elif p.type == PartType.input_video: raise ValueError("OpenAI Chat API does not support video input")
        elif p.type == PartType.input_file:  parts.append(_denorm_oai_chat_file(p))
    if len(parts) == 1 and parts[0].get('type') == 'text': return dict(role='user', content=parts[0]['text'])
    return dict(role='user', content=parts)

def denorm_anthropic_user(m:Msg):
    "Convert canonical user Msg to Anthropic user message."
    parts = []
    for p in m.content:
        if   p.type == PartType.text:        parts.append(_ant_cc({"type": "text", "text": p.text or ""}, p))
        elif p.type == PartType.input_image: parts.append(_ant_cc(_denorm_ant_image(p), p))
        elif p.type == PartType.input_audio: raise ValueError("Anthropic does not support audio input")
        elif p.type == PartType.input_video: raise ValueError("Anthropic does not support video input")
        elif p.type == PartType.input_file:  parts.append(_ant_cc(_denorm_ant_file(p), p))
    return dict(role='user', content=parts)

def denorm_gemini_user(m:Msg):
    "Convert canonical user Msg to Gemini user content."
    parts = []
    for p in m.content:
        if   p.type == PartType.text:        parts.append({"text": p.text or ""})
        elif p.type == PartType.input_image: parts.append(_denorm_gem_image(p))
        elif p.type == PartType.input_audio: parts.append(_denorm_gem_audio(p))
        elif p.type == PartType.input_video: parts.append(_denorm_gem_video(p))
        elif p.type == PartType.input_file:  parts.append(_denorm_gem_file(p))
    return dict(role='user', parts=parts)

#### Image

In [ ]:
#| export
def _denorm_oai_resp_image(p): return {"type": "input_image", "image_url": p.text}
def _denorm_oai_chat_image(p): return {"type": "image_url", "image_url": {"url": p.text}}
def _denorm_ant_image(p):
    if (b64:=_data_url(p.text)): return {"type": "image", "source": {"type": "base64", "media_type": b64[0], "data": b64[1]}}
    return {"type": "image", "source": {"type": "url", "url": p.text}}
def _denorm_gem_image(p):
    if (b64:=_data_url(p.text)): return {"inlineData": {"mimeType": b64[0], "data": b64[1]}}
    return {"fileData": {"mimeType": _url_mime(p.text, "image/*"), "fileUri": p.text}}

In [ ]:
img_url = "https://img.freepik.com/free-photo/mountain-range-body-water_53876-139760.jpg?semt=ais_hybrid&w=740&q=80"
for p in provs:
    msg1 = Msg(role='user', content=[Part(type=PartType.input_image, text=img_url), Part(type=PartType.text, text='What is this image?')])
    comp = await stream_complete([msg1], model=models[p], prov=p)
    print(f'  {p.value:12}: {comp.message.content[0].text[:80]}')

In [ ]:
img_b64 = "data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAQAAAAECAIAAAAmkwkpAAAAEElEQVR4nGP4z8AARwzEcQCukw/x0F8jngAAAABJRU5ErkJggg=="
for p in provs:
    msg1 = Msg(role='user', content=[Part(type=PartType.input_image, text=img_b64), Part(type=PartType.text, text='What color is this pixel?')])
    comp = await stream_complete([msg1], model=models[p], prov=p)
    print(f'  {p.value:12}: {comp.message.content[0].text[:80]}')

#### Audio

https://developers.openai.com/api/docs/guides/migrate-to-responses#responses-benefits - responses api support coming soon

In [ ]:
#| export
_mime_audio_fmt = {'audio/wav':'wav', 'audio/mpeg':'mp3', 'audio/mp3':'mp3'}
def _denorm_oai_chat_audio(p):
    if not (b64:=_data_url(p.text)): raise ValueError("OpenAI Chat audio input requires base64 data URL")
    return {"type": "input_audio", "input_audio": {"data": b64[1], "format": _mime_audio_fmt.get(b64[0], 'wav')}}
def _denorm_gem_audio(p): 
    if (b64:=_data_url(p.text)): return {"inlineData": {"mimeType": b64[0], "data": b64[1]}}
    return {"fileData": {"mimeType": _url_mime(p.text, "audio/*"), "fileUri": p.text}}

In [ ]:
wav_data = httpx.get("https://openaiassets.blob.core.windows.net/$web/API/docs/audio/alloy.wav").content
audio_b64 = f"data:audio/wav;base64,{base64.b64encode(wav_data).decode()}"

OpenAI chat requires a [specific model](https://developers.openai.com/api/docs/guides/audio?example=audio-in#add-audio-to-your-existing-application)

In [ ]:
models[ApiName.openai_chat] = 'gpt-4o-audio-preview'

OpenAI Responses and Anthropic don't support audio

In [ ]:
audio_provs = [ApiName.openai_chat, ApiName.gemini]
for p in audio_provs:
    msg1 = Msg(role='user', content=[Part(type=PartType.input_audio, text=audio_b64), Part(type=PartType.text, text='What is this audio saying?')])
    comp = await stream_complete([msg1], model=models[p], prov=p)
    print(f'  {p.value:12}: {comp.message.content[0].text[:80]}')

#### Video

In [ ]:
#| export
def _denorm_gem_video(p):
    if (b64:=_data_url(p.text)): return {"inlineData": {"mimeType": b64[0], "data": b64[1]}}
    return {"fileData": {"mimeType": _url_mime(p.text, "video/mp4"), "fileUri": p.text}}

Only supported by Gemini models

In [ ]:
vid_url = "https://storage.googleapis.com/cloud-samples-data/video/animals.mp4"
msg1 = Msg(role='user', content=[Part(type=PartType.input_video, text=vid_url), Part(type=PartType.text, text='Concisely, what animals are in this video?')])
comp = await stream_complete([msg1], model=models[ApiName.gemini], prov=ApiName.gemini)
print(f'  gemini (url)    : {comp.message.content[0].text[:120]}')

In [ ]:
vid_yt = "https://www.youtube.com/watch?v=9hE5-98ZeCg"
msg1 = Msg(role='user', content=[Part(type=PartType.input_video, text=vid_yt), Part(type=PartType.text, text='Summarize this video in one sentence.')])
comp = await stream_complete([msg1], model=models[ApiName.gemini], prov=ApiName.gemini)
print(f'  gemini (youtube): {comp.message.content[0].text[:120]}')

#### Files

In [ ]:
models[ApiName.openai_chat] = 'gpt-4o-mini'

OpenAI requires a filename, Anthropic requires the media type and Gemini requires the mime type

In [ ]:
#| export
def _denorm_oai_resp_file(p):
    if (b64:=_data_url(p.text)): return {"type": "input_file", "file_data": p.text, "filename": f"upload.{b64[0].split('/')[-1]}"}
    return {"type": "input_file", "file_url": p.text}

def _denorm_oai_chat_file(p):
    if (b64:=_data_url(p.text)): return {"type": "file", "file": {"file_data": p.text, "filename": f"upload.{b64[0].split('/')[-1]}"}}
    raise ValueError("OpenAI Chat file input requires base64 data URL or file_id, not URLs")

def _denorm_ant_file(p):
    if (b64:=_data_url(p.text)): return {"type": "document", "source": {"type": "base64", "media_type": b64[0], "data": b64[1]}}
    return {"type": "document", "source": {"type": "url", "url": p.text}}

def _denorm_gem_file(p):
    if (b64:=_data_url(p.text)): return {"inlineData": {"mimeType": b64[0], "data": b64[1]}}
    return {"fileData": {"mimeType": _url_mime(p.text, "application/pdf"), "fileUri": p.text}}

In [ ]:
pdf_url = "https://arxiv.org/pdf/1706.03762"
for p in [ApiName.openai, ApiName.anthropic, ApiName.gemini]:
    msg1 = Msg(role='user', content=[Part(type=PartType.input_file, text=pdf_url), Part(type=PartType.text, text='What is the title of this paper?')])
    comp = await stream_complete([msg1], model=models[p], prov=p)
    print(f'  {p.value:12}: {comp.message.content[0].text[:80]}')

In [ ]:
pdf_b64 = f"data:application/pdf;base64,{base64.b64encode(httpx.get(pdf_url).content).decode()}"
for p in provs:
    msg1 = Msg(role='user', content=[Part(type=PartType.input_file, text=pdf_b64), Part(type=PartType.text, text='What is the title of this paper?')])
    comp = await stream_complete([msg1], model=models[p], prov=p)
    print(f'  {p.value:12}: {comp.message.content[0].text[:80]}')

### Media Tool Call Results

A tool call can return an `input_image` or `input_file`

In [ ]:
#| export
def denorm_openai_responses_tool_result(m:Part):
    "Convert canonical tool result back to OpenAI Responses function_call_output item."
    cid = m.data.get('id', '') or m.data.get('call_id')
    if isinstance(m.text, list):
        out = []
        for p in m.text:
            if   p.type == PartType.text:        out.append({"type": "input_text", "text": p.text or ""})
            elif p.type == PartType.input_image: out.append(_denorm_oai_resp_media(p))
            elif p.type == PartType.input_file:  out.append(_denorm_oai_resp_file(p))
            else: raise ValueError(f"OpenAI Responses tool_result does not support {p.type}")
        return dict(type='function_call_output', call_id=cid, output=out)
    return dict(type='function_call_output', call_id=cid, output=str(m.text))

In [ ]:
#| export
def denorm_openai_chat_tool_result(p:Part):
    "Convert canonical tool_result Part to OpenAI Chat tool message."
    if isinstance(p.text, list): raise ValueError("OpenAI Chat does not support media in tool results")
    return dict(role='tool', tool_call_id=p.data.get('id') or p.data.get('call_id', ''), content=str(p.text))

In [ ]:
#| export
def denorm_anthropic_tool_result(p:Part):
    "Convert canonical tool_result Part to Anthropic tool_result content block."
    d = p.data or {}
    tid = d.get('id') or d.get('call_id','')
    if isinstance(p.text, list):
        blocks = []
        for pp in p.text:
            if   pp.type == PartType.text:        blocks.append({"type": "text", "text": pp.text or ""})
            elif pp.type == PartType.input_image: blocks.append(_denorm_ant_media(pp))
            elif pp.type == PartType.input_file:  blocks.append(_denorm_ant_file(pp))
            else: raise ValueError(f"Anthropic tool_result does not support {pp.type}")
        return _ant_cc(dict(type='tool_result', tool_use_id=tid, content=blocks), p)
    return _ant_cc(dict(type='tool_result', tool_use_id=tid, content=str(p.text)), p)

In [ ]:
#| export
def denorm_gemini_tool_result(p:Part):
    "Convert canonical tool_result Part to Gemini functionResponse part."
    d = p.data or {}
    fr = dict(name=d.get('name',''), response={"content": "" if isinstance(p.text, list) else str(p.text)})
    if d.get('id'): fr['id'] = d['id']
    if isinstance(p.text, list):
        parts = []
        for pp in p.text:
            if   pp.type == PartType.text:        parts.append({"text": pp.text or ""})
            elif pp.type == PartType.input_image: parts.append(_denorm_gem_media(pp))
            elif pp.type == PartType.input_file:  parts.append(_denorm_gem_file(pp))
            else: raise ValueError(f"Gemini tool_result does not support {pp.type}")
        fr['parts'] = parts
    return dict(functionResponse=fr)

In [ ]:
msgs = [Msg(role='user', content=[Part(type=PartType.text, text="What's in this image?")]),
        Msg(role='assistant', content=[Part(type=PartType.tool_use, data={'id': '_test', 'name': 'test_img', 'arguments': {}, 'server': False})]),
        Msg(role='tool', content=[Part(type=PartType.tool_result, text=[Part(type=PartType.input_image, text=img_b64)], data={'id': '_test', 'name': 'test_img'})])]

In [ ]:
for p in provs:
    try:
        comp = await stream_complete(msgs, model=models[p], prov=p)
        print(models[p], comp.message.content[0].text, '\n')
    except Exception as e: print(e, '\n')

### Infer Client

Utils to infer client setup from model name. Users can just pass model name to `acomplete` and the provider will be inferred and client will be create automatically

In [ ]:
#| export
vendor_mapping = {
    "openai":       ('https://api.openai.com/v1', 'OPENAI_API_KEY'), # to explicitly choose responses/chat api
    "moonshot":     ("https://api.moonshot.ai/v1", "MOONSHOT_API_KEY"),
    "deepseek":     ("https://api.deepseek.com/v1", "DEEPSEEK_API_KEY"),
    "openrouter":   ("https://openrouter.ai/api/v1", "OPENROUTER_API_KEY"),
    "together":     ("https://api.together.xyz/v1", "TOGETHER_API_KEY"),
    "fireworks_ai": ("https://api.fireworks.ai/inference/v1", "FIREWORKS_API_KEY"),
    "qwen":         ("https://dashscope.aliyuncs.com/compatible-mode/v1", "QWEN_API_KEY")
}

In [ ]:
#| export
def infer_api_name(model):
    "Infer api_name from model"
    if "claude" in model: return ApiName.anthropic
    if "gemini" in model: return ApiName.gemini
    if "gpt" in model:    return ApiName.openai
    return ApiName.openai_chat

In [ ]:
#| export
@flexicache()
def mk_client(model, api_name=None, vendor_name=None, env_api_key=None, base_url=None, xtra_hdrs={}):
    api_name = ifnone(api_name, infer_api_name(model))
    if api_name == ApiName.openai:      spec, hdrs, vendor_name = oai_spec, {"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"}, 'openai'
    elif api_name == ApiName.anthropic: spec, hdrs, vendor_name = ant_spec, {"x-api-key": os.environ['ANTHROPIC_API_KEY'], "anthropic-version": "2023-06-01"}, 'anthropic'
    elif api_name == ApiName.gemini:    spec, hdrs, vendor_name = gem_spec, {"x-goog-api-key": os.environ['GEMINI_API_KEY']}, 'gemini'
    elif api_name == ApiName.openai_chat:
        valid_names = ', '.join(list(vendor_mapping.keys()))
        if not (env_api_key and base_url):
            if not vendor_name: raise ValueError(f"Model: '{model}' can't be auto inferred please pass a valid `vendor_name` : {valid_names} or `env_api_key` and `base_url`")
            try: base_url, env_api_key = vendor_mapping[vendor_name]
            except: raise ValueError(f"Vendor '{vendor_name}' can't be auto inferred please pass a valid one : {valid_names} or pass `env_api_key` and `base_url`")
        cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ[env_api_key]}"})
        for op in cli.ops: op.base_url = base_url
        return cli, api_name, vendor_name
    return OpenAPIClient(spec, headers=merge(hdrs, xtra_hdrs)), api_name, vendor_name

In [ ]:
models[ApiName.openai_chat]

In [ ]:
print(mk_client(models[ApiName.openai])[0].ops[0].base_url)
# to force openai chat
print(mk_client(models[ApiName.openai_chat], api_name=ApiName.openai_chat, env_api_key='OPENAI_API_KEY', base_url='https://api.openai.com/v1')[0].ops[0].base_url)
print(mk_client(models[ApiName.anthropic])[0].ops[0].base_url)
print(mk_client(models[ApiName.gemini])[0].ops[0].base_url)

In [ ]:
print(mk_client('kimi-k2.5', vendor_name='moonshot')[0].ops[0].base_url)

In [ ]:
try: mk_client('my-custom-model')
except Exception as e: print(e)

In [ ]:
try: mk_client('my-custom-model', vendor_name='xxx')
except Exception as e: print(e)

## Acomplete

### Error Classification

Different providers signal "context window exceeded" in different ways — OpenAI uses `code: "context_length_exceeded"`, while Anthropic and Gemini embed hints in the error message (e.g. "prompt is too long", "exceeds the maximum number of tokens allowed"). We classify these into a dedicated `ContextWindowExceededError` subclass so callers (e.g. lisette's tool loop) can catch and recover specifically.

- **OpenAI** (Chat & Responses): HTTP 400 with `error.type = "invalid_request_error"` and `error.code = "context_length_exceeded"`. Message starts with "This model's maximum context length is …". See [OpenAI docs troubleshooting](https://devidevs.com/blog/openai-api-errors-troubleshooting).
- **Anthropic**: HTTP 400 with `error.type = "invalid_request_error"` and message `"prompt is too long: N tokens > M maximum"`. See [continuedev/continue#6270](https://github.com/continuedev/continue/issues/6270).
- **Gemini**: HTTP 400 with `error.status = "INVALID_ARGUMENT"` and message `"The input token count (N) exceeds the maximum number of tokens allowed (M)."`. See [gemini-cli#11939](https://github.com/google-gemini/gemini-cli/issues/11939).

Substring list here is derived from [litellm's `is_error_str_context_window_exceeded`](https://github.com/BerriAI/litellm/blob/main/litellm/litellm_core_utils/exception_mapping_utils.py).

In [ ]:
#| export
class ContextWindowExceededError(APIError): pass

def _is_ctx_exceeded(code, msg):
    m = (msg or "").lower()
    if any(x in m for x in ("string_above_max_length", "invalid 'user'")): return False
    if str(code or "").lower() == "context_length_exceeded": return True
    return any(s in m for s in ("exceed context limit", "maximum context length", "maximum context limit",
    "longer than the model's context length", "input tokens exceed the configured limit",
    "exceeds the maximum number of tokens allowed", "prompt is too long"))

def _classify_error(exc):
    "Upgrade generic `APIError` to a specific subclass if applicable."
    if not isinstance(exc, APIError): return exc
    if _is_ctx_exceeded(exc.code, exc.message):
        return ContextWindowExceededError(exc.message, provider=exc.provider, model=exc.model,
            endpoint=exc.endpoint, status_code=exc.status_code, error_type=exc.error_type,
            code=exc.code, request_id=exc.request_id, retryable=exc.retryable, raw=exc.raw)
    return exc

async def _classify_error_stream(gen):
    "Wrap an async generator to upgrade `APIError`s as they're raised during iteration."
    try:
        async for x in gen: yield x
    except APIError as e: raise _classify_error(e) from e

In [ ]:
e_oai = APIError("This model's maximum context length is 4097 tokens...", status_code=400,
                 error_type="invalid_request_error", code="context_length_exceeded")
e_ant = APIError("prompt is too long: 210503 tokens > 200000 maximum", status_code=400,
                 error_type="invalid_request_error")
e_gem = APIError("The input token count (1632254) exceeds the maximum number of tokens allowed (1048576).",
                 status_code=400, error_type="INVALID_ARGUMENT")

test(_classify_error(e_oai), ContextWindowExceededError, isinstance)
test(_classify_error(e_ant), ContextWindowExceededError, isinstance)
test(_classify_error(e_gem), ContextWindowExceededError, isinstance)

In [ ]:
#| export
async def _send(func, payload, norm, stream, model, api_name, vendor_name):
    "Await `func(**payload)`, classify errors, and either wrap stream or normalize response."
    try: resp = await func(**payload)
    except APIError as e: raise _classify_error(e) from e
    if stream: return _classify_error_stream(acollect_stream(resp, model=model, api_name=api_name, vendor_name=vendor_name))
    return norm(resp, model=model, api_name=api_name, vendor_name=vendor_name)

### acomplete()

In [ ]:
#| export
def _sys_text(system):
    "Extract text from system (str or Part)."
    if system is None: return None
    return system if isinstance(system, str) else system.text

async def acomplete(msgs, model, stream=False, system=None, max_tokens=None, temperature=None, tools=None, tool_choice=None, reasoning_effort=None, web_search_options=None, api_name=None, vendor_name=None, env_api_key=None, base_url=None, xtra_hdrs={}):
    "Unified completion across different APIs."
    cli, api_name, vendor_name = mk_client(model, api_name, vendor_name, env_api_key, base_url, xtra_hdrs)
    if api_name == ApiName.openai:
        payload = dict(model=model, input=denorm_openai_responses_msgs(msgs))
        if stream:                  payload['stream'] = True
        if system:                  payload['instructions'] = _sys_text(system)
        if max_tokens:              payload['max_output_tokens'] = max_tokens
        if temperature is not None: payload['temperature'] = temperature
        if tools:                   payload['tools'] = denorm_openai_responses_tools(tools)
        if web_search_options is not None: payload.setdefault('tools', []).append(denorm_openai_responses_web_search(web_search_options))
        if tool_choice:      payload['tool_choice'] = denorm_openai_responses_tool_choice(tool_choice)
        if reasoning_effort: payload['reasoning'] = denorm_openai_responses_reasoning(reasoning_effort)
        return await _send(cli.responses.create_response, payload, normalize_openai_response, stream, model, api_name, vendor_name)

    elif api_name == ApiName.openai_chat:
        payload = dict(model=model, messages=denorm_openai_chat_msgs(msgs))
        if system:           payload['messages'].insert(0, {'role': 'system', 'content': _sys_text(system)})
        if stream: payload.update(stream=True, stream_options={"include_usage": True})
        if max_tokens:       payload['max_tokens'] = max_tokens
        if temperature is not None: payload['temperature'] = temperature
        if tools:            payload['tools'] = denorm_openai_chat_tools(tools)
        if web_search_options is not None: payload['web_search_options'] = denorm_openai_chat_web_search(web_search_options)
        if tool_choice:      payload['tool_choice'] = denorm_openai_chat_tool_choice(tool_choice)
        if reasoning_effort: payload['reasoning_effort'] = denorm_openai_chat_reasoning(reasoning_effort)
        return await _send(cli.chat.create_chat_completion, payload, normalize_openai_chat_completion, stream, model, api_name, vendor_name)

    elif api_name == ApiName.anthropic:
        payload = dict(model=model, messages=denorm_anthropic_msgs(msgs), max_tokens=max_tokens or 1024)
        if stream: payload['stream'] = True
        if system:
            if isinstance(system, Part):
                block = {'type': 'text', 'text': system.text or ''}
                if (cc := (system.data or {}).get('cache_control')): block['cache_control'] = cc
                payload['system'] = [block]
            else: payload['system'] = system
        if temperature is not None: payload['temperature'] = temperature
        if tools:            payload['tools'] = denorm_anthropic_tools(tools)
        if web_search_options is not None: payload.setdefault('tools', []).append(denorm_anthropic_web_search(web_search_options))
        tc = denorm_anthropic_tool_choice(tool_choice)
        if tc:               payload['tool_choice'] = tc
        think = denorm_anthropic_reasoning(reasoning_effort)
        if think: payload.update(think)
        return await _send(cli.messages.messages_post, payload, normalize_anthropic_message, stream, model, api_name, vendor_name)

    elif api_name == ApiName.gemini:
        gen_config = {}
        if max_tokens:       gen_config['maxOutputTokens'] = max_tokens
        if temperature is not None: gen_config['temperature'] = temperature
        think = denorm_gemini_reasoning(reasoning_effort,model)
        if think:            gen_config['thinkingConfig'] = think
        payload = dict(model=model, contents=denorm_gemini_msgs(msgs))
        if system:           payload['system_instruction'] = {'parts': [{'text': _sys_text(system)}]}
        if gen_config:       payload['generation_config'] = gen_config
        gem_tools = denorm_gemini_tools(tools) if tools else []
        if web_search_options is not None: gem_tools.append(denorm_gemini_web_search(web_search_options))
        if gem_tools:
            payload['tools'] = gem_tools
            has_fn = any('functionDeclarations' in t for t in gem_tools)
            has_srv = any(k in t for t in gem_tools for k in ('googleSearch','codeExecution','googleSearchRetrieval'))
            if has_fn and has_srv:
                payload.setdefault('tool_config', {})['includeServerSideToolInvocations'] = True
        tc = denorm_gemini_tool_choice(tool_choice)
        if tc:               payload.setdefault('tool_config', {}).update(tc)
        op = 'stream_generate_content' if stream else 'generate_content'
        if stream:           payload.update(stream=True, _query={"alt": "sse"})
        return await _send(getattr(cli.models, op), payload, normalize_gemini_generate, stream, model, api_name, vendor_name)

In [ ]:
@delegates(acomplete)
async def print_stream(msgs, model, **kwargs):
    cnt,max_print = 0,10
    async for o in await acomplete(msgs, model, stream=True, **kwargs): 
        if not isinstance(o, Completion): 
            if o.get('thinking') and cnt<max_print: print('🧠',end='',flush=True)
            if (txt:=o.get('text')): print(txt,end='',flush=True)
            cnt+=1
    return o

In [ ]:
msg1 = Msg(role='user', content=[Part(type=PartType.text, text='Hello!')])

In [ ]:
comp = await print_stream([msg1], model='gpt-4o-mini'); comp.usage

In [ ]:
comp = await print_stream([msg1], model='gpt-4o-mini', api_name=ApiName.openai_chat, env_api_key='OPENAI_API_KEY', base_url='https://api.openai.com/v1'); comp.usage

In [ ]:
comp = await print_stream([msg1], model='claude-sonnet-4-20250514'); comp.usage

In [ ]:
comp = await print_stream([msg1], model='models/gemini-3-flash-preview'); comp.usage

In [ ]:
comp = await print_stream([msg1], model='kimi-k2.5', vendor_name='moonshot'); comp.usage

In [ ]:
comp = await print_stream([msg1], model='fireworks/kimi-k2p5', vendor_name='fireworks_ai'); comp.usage

In [ ]:
comp.usage, comp.api_name, comp.vendor_name

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()